In [1]:
# Imports for the modeling notebook.
import warnings
warnings.filterwarnings('ignore')

from scipy.stats import norm
import biogeme.biogeme as bio
from biogeme.expressions import bioDraws, log, Elem
from biogeme import models
from biogeme.models import ordered_probit, ordered_logit
from biogeme import results as res
import pandas as pd
import biogeme.database as db
import pickle


In [2]:
# Load the cleaned dataset produced by `loading_and_cleaning_data.ipynb`.
# low_memory=False suppresses the mixed-type DtypeWarning on the wide CSV.
df = pd.read_csv('final_processed_crash_dataset.csv', low_memory=False)


## Sanity check and preparing the databases

In [3]:
df=df[df['severity']!=-1]
df=df.loc[df['Vehicle'].isin(['E-PMD','Bike','E-bike','Pedestrian'])]

df_dummies=pd.get_dummies(df[['Crossroad','Helmet','Point of impact','Gender','vehicle_type_2','Vehicle','Pavement','Intersection','vehicle_type_3'
                              , 'User category', 'Lighting conditions', 'Cycle facilities', 'Road width', 'Agglomeration', 'Age category',
                              'Accident location', 'Surface condition', 'Maneuver', 'Maneuver_2','Gender_2', 'Pedestrian localisation', 'Pedestrian action', 'Vehicle_2',
'Max speed', 'Long profile', 'Weather conditions', 'Road type', 'Trip purpose','Reflective jacket', 'plan','Point of impact_2', 'Obstacle','Gender_driver','Helmet_driver','age_driver','Year', 'Maneuver_3','Point of impact_3','Gender_3','Vehicle_3'

]])
df_dummies = df_dummies.astype(int)  # Convert boolean to integers

continuous_vars = [
    'age',
    'Number of passengers',
    'number of involved vehicles',
    'vma',
    'age_2',
    'age_opposite_mean'
]

df_centered = df.copy()
# cellule 3, avant le centrage
df_centered['vma_raw'] = df['vma']
df_centered['Number of passengers raw'] = df['Number of passengers']


# `age_opposite_mean == 0` n'est pas un age : c'est le code des collisions sans
# tiers identifie (2 212 des 2 214 lignes du segment sans tiers). On le traite
# comme manquant AVANT de centrer, sinon ces zeros tirent la moyenne vers le bas
# et decalent toute la variable.
df_centered['age_opposite_mean'] = df_centered['age_opposite_mean'].mask(
    df_centered['age_opposite_mean'] == 0
)

df_centered[continuous_vars] = (
    df_centered[continuous_vars]
    - df_centered[continuous_vars].mean()
)


df_non_dummies = df_centered[['age','severity','Number of passengers','number of involved vehicles','vma','vma_raw','age_2','catu', 'Num_Acc','age_opposite_mean']]




In [4]:
df_non_dummies=pd.concat([df_non_dummies,df_dummies],axis=1)

## Removing useless rows
df_non_dummies=df_non_dummies.dropna(subset=['age','severity'])


# On garde les observations sans tiers identifie au lieu de les supprimer :
# la variable est mise a 0 -- soit la moyenne, puisque les continues sont
# centrees -- et l'indicatrice ci-dessous absorbe le niveau propre de ce groupe.
# `beta_age_opposite_mean` reste donc identifie sur les seules lignes renseignees.
df_non_dummies['age_opposite_missing'] = (
    df_non_dummies['age_opposite_mean'].isna().astype(int)
)
df_non_dummies['age_opposite_mean'] = df_non_dummies['age_opposite_mean'].fillna(0.0)

df_non_dummies = df_non_dummies.reset_index(drop=True)

In [5]:
## First model
df_carcrashes=df_non_dummies.loc[(df_non_dummies['vehicle_type_2_Cars']==1) | (df_non_dummies['vehicle_type_2_Large motorized vehicle'] ==1) | (df_non_dummies['vehicle_type_2_Light motorized vehicle']==1) ]
df_carcrashes=df_carcrashes.loc[df_carcrashes['catu'].isin([1,2])]
database_carcrashes = db.Database('database_carcrashes', df_carcrashes)

## Second model
df_mmv=df_non_dummies.loc[(df_non_dummies['vehicle_type_2_Micromobility vehicle']==1) ]
df_mmv=df_mmv.loc[df_mmv['catu'].isin([1,2])]
# L'unique observation mortelle est CONSERVEE : le modele regroupe deces et
# blessure via `harmed`, donc un deces isole n'a plus besoin d'etre identifie
# separement. La supprimer ferait aussi diverger cet echantillon de celui des
# statistiques descriptives, qui le conservent.
df_mmv = df_mmv.copy()  # age du tiers manquant : conserve, via age_opposite_missing

database_mmv= db.Database('database_mmv',df_mmv)


## Third model
num_acc_values = df_non_dummies.loc[df_non_dummies['vehicle_type_2_Pedestrian'] == 1, 'Num_Acc']
df_pedestrian = df_non_dummies[df_non_dummies['Num_Acc'].isin(num_acc_values)]
df_pedestrian = df_pedestrian.copy()  # idem : plus aucune ligne supprimee ici

database_pedestrian= db.Database('database_pedestrian',df_pedestrian)

## Fourth model
df_sv=df_non_dummies.loc[df_non_dummies['vehicle_type_2_No other vehicle']==1]
df_sv=df_sv.loc[df_sv['catu'].isin([1,2])]
database_sv= db.Database('database_sv',df_sv)


## Variables and Betas

In [6]:
import re
import biogeme.database as db
from biogeme.expressions import Beta, Variable


def normalize(name: str) -> str:
    clean = name.lower()
    clean = re.sub(r'[^a-z0-9]+', '_', clean)
    clean = re.sub(r'_+', '_', clean)
    clean = clean.strip('_')
    if re.match(r'^[0-9]', clean):
        clean = "var_" + clean
    return clean

base_suffixes = ['','_I', '_F', '_mean', '_I_mean', '_F_mean','_sd','_I_std','_F_std']
modes = ['', '_epmd', '_bike', '_ebike']  

for col in df_non_dummies.columns:
    clean_name = normalize(col)

    # Skip si nettoyage donne v_injuryde
    if clean_name == "":
        print(f"Skip (empty after cleaning): {col}")
        continue

    if clean_name not in locals():
        exec(f"{clean_name} = Variable({repr(col)})")
    # Pour chaque suffixe de base et chaque mode, créer un Beta si non existant
    for suf in base_suffixes:
        for mode in modes:
            beta_var_name = f"beta_{clean_name}{suf}{mode}"
            beta_label = beta_var_name

            if '_std' in suf or suf == '_std':
                init_value = 1
            else:
                init_value = 0

            if not beta_var_name.isidentifier():
                print(f"Skip beta (invalid identifier): {beta_var_name}")
                continue

            if beta_var_name in locals():
                continue
            exec(f"{beta_var_name} = Beta({repr(beta_label)}, {init_value}, None, None, 0)")

constant_I = Beta('constant_I', 0, None, None, 0)
constant_F = Beta('constant_F', 0, None, None, 0)

# Reponse binaire : 1 indemne, 2 blesse ou tue.
#
# Certains segments ne comptent pas assez de deces pour identifier un
# parametre qui leur soit propre -- 9 chez les pietons, 1 dans les collisions
# engin-engin. Plutot que d'ecarter ces observations, on regroupe les deux
# issues nefastes. La fusion porte sur la VARIABLE EXPLIQUEE, jamais sur les
# utilites : donner la meme utilite a deux alternatives ne les fusionne pas,
# elle contraint leurs probabilites a etre egales, ce que les donnees
# dementent violemment.
harmed = (severity > 1) + 1
availability_harmed = {1: 1, 2: 1}


In [7]:
# --- Where each model writes its results --------------------------------------
# Biogeme writes `<modelName>.html`, `<modelName>.pickle` and the iterations file
# `__<modelName>.iter` in the current working directory, and exposes no parameter
# to redirect them. Putting a path in `modelName` is not an option: it would also
# become the name displayed inside the results, and it would turn the iterations
# file into an invalid path. Changing directory around the call is the only way.

import os
from contextlib import contextmanager
from pathlib import Path

RESULTS_ROOT = Path('results')
RESULTS_DIRECTORIES = {
    'carcrashes': RESULTS_ROOT / 'model_carcrashes',
    'mmv': RESULTS_ROOT / 'model_mmv',
    'pedestrian': RESULTS_ROOT / 'model_pedestrian',
    'single_vehicle': RESULTS_ROOT / 'model_single_vehicle',
}
for directory in RESULTS_DIRECTORIES.values():
    directory.mkdir(parents=True, exist_ok=True)


@contextmanager
def _in_results_directory(segment):
    """Temporarily make `segment`'s directory the working directory.

    The Biogeme object must already be built when this is used: it reads
    `biogeme.toml` from the working directory at construction time, so building
    it inside the sub-directory would silently fall back to default parameters.
    """
    previous = Path.cwd()
    os.chdir(RESULTS_DIRECTORIES[segment])
    try:
        yield
    finally:
        os.chdir(previous)


def save_results(results, segment, model_name):
    """Write an already-estimated model's HTML and pickle into `segment`'s directory.

    Used for the models that are estimated through a helper rather than directly,
    which switch Biogeme's own output off so that intermediate fits do not
    produce files. Any previous file of the same name is removed first: Biogeme
    otherwise appends a ~00, ~01, ... suffix at every run, which is what filled
    the repository root with hundreds of files.
    """
    results.data.modelName = model_name
    with _in_results_directory(segment):
        for extension in ('html', 'pickle'):
            Path(f'{model_name}.{extension}').unlink(missing_ok=True)
        results.write_html(True)
        results.write_pickle()
    return results


def estimate_in(segment, the_biogeme, **kwargs):
    """Estimate a model, writing its HTML/pickle output into `segment`'s directory."""
    with _in_results_directory(segment):
        return the_biogeme.estimate(**kwargs)


def validate_in(segment, the_biogeme, estimation_results, validation_data):
    """Out-of-sample validation, writing its output into `segment`'s directory."""
    with _in_results_directory(segment):
        return the_biogeme.validate(estimation_results, validation_data)

## Model for car crashes

### Baseline (constants only)

We first estimate the constants-only model. Its log-likelihood is the
denominator used for the rho-square statistics reported below and for the
likelihood-ratio test (see the LR-test section).


In [8]:
## Constant model

availability = {
    1: 1, 
    2: 1, 
    3: 1   
}

U={1:0,2:constant_I,3:constant_F}

model_name = 'InitialModel_carcrashes'


logprob = models.loglogit(U, availability, severity)

# Créez l'objet Biogeme
model_cst_car = bio.BIOGEME(database_carcrashes, logprob)
model_cst_car.modelName = model_name

# Estimation


results_constant_car = estimate_in('carcrashes', model_cst_car)



### Full mixed-logit specification


In [9]:
# `Maneuver_2` / `vehicle_type_2` decrivent le premier tiers, `_3` le second.
# Les sommer donnerait 2 quand les deux tiers partagent la caracteristique : on
# veut une indicatrice « au moins un des tiers », donc un OU logique. Sur ce
# segment, 35 collisions ont un vehicule motorise leger en tiers 3 sans l'avoir
# en tiers 2, et une seule les deux.
def any_of(*terms):
    """Indicatrice 0/1 : au moins un des termes est non nul."""
    total = terms[0]
    for term in terms[1:]:
        total = total + term
    return total > 0


v_no_injury = 0

v_injury = (  beta_gender_female_I * gender_female  + constant_I +beta_age_I*age
      + beta_user_category_passenger_I * user_category_passenger
     + beta_number_of_involved_vehicles_I * number_of_involved_vehicles
      + beta_point_of_impact_back_I * point_of_impact_back
      + beta_vehicle_type_2_light_motorized_vehicle_I * any_of(vehicle_type_2_light_motorized_vehicle,
                                                             vehicle_type_3_light_motorized_vehicle)
    # + beta_maneuver_2_overtaking_I * any_of(maneuver_2_overtaking, maneuver_3_overtaking)
      + beta_maneuver_without_change_of_direction_I * maneuver_without_change_of_direction
    #  + user_category_passenger * (beta_gender_driver_female_I * gender_driver_female)
     + beta_intersection_no_intersection_I * (intersection_no_intersection)
 + beta_maneuver_2_overtaking_I * any_of(maneuver_2_overtaking, maneuver_3_overtaking)
)

v_fatality = (  constant_F
     
        + beta_age_F * age

        + beta_vma_F * vma * intersection_no_intersection
        + beta_vehicle_type_2_large_motorized_vehicle_F * any_of(vehicle_type_2_large_motorized_vehicle,
                                                                vehicle_type_3_large_motorized_vehicle)
                                                           
        + beta_lighting_conditions_night_with_street_lightings_on_F * (lighting_conditions_night_with_street_lightings_on + lighting_conditions_night_without_street_lightings)
        + beta_maneuver_2_turning_right_F * any_of(maneuver_2_turning_right, maneuver_3_turning_right)
        + beta_accident_location_on_cycle_facility_F* accident_location_on_cycle_facility
       # + beta_lighting_conditions_night_without_street_lightings_F* lighting_conditions_night_without_street_lightings
   #    + user_category_passenger * (beta_gender_driver_female_I * gender_driver_female)
    # + beta_intersection_no_intersection_F  * intersection_no_intersection
    
)

utility_motorized_vehicles = {
    1: v_no_injury,
    2: v_injury,
    3: v_fatality
}


In [10]:
# Random-parameters model
sigma_I = Beta('sigma I', 0, None, None, 0)

X1 = bioDraws('X1', 'NORMAL')


# Adding the error component to the utilities
v_no_injury_rp=0
v_injury_rp = utility_motorized_vehicles[2] 
v_fatality_rp = utility_motorized_vehicles[3]

utility_motorized_vehicles_mixed={1:v_no_injury_rp,2:v_injury_rp,3:v_fatality_rp}

prob = models.logit(utility_motorized_vehicles_mixed,availability,severity)


# We integrate over B_TIME_RND using Monte-Carlo
logprob = log((prob))



# Create the Biogeme object
model_car  = bio.BIOGEME(database_carcrashes,logprob)
model_car.modelName = "logit_car_crashes"

# Estimate the parameters. 
results_ml_carcrashes = estimate_in('carcrashes', model_car)


In [11]:
results_ml_carcrashes.get_estimated_parameters()

,Value,Rob. Std err,Rob. t-test,Rob. p-value
beta_accident_location_on_cycle_facility_F,-0.991482,0.516705,-1.918854,5.500284e-02
beta_age_F,0.054596,0.009245,5.905343,3.519145e-09
beta_age_I,0.015889,0.004568,3.478468,5.042894e-04
beta_gender_female_I,0.639220,0.135531,4.716420,2.400310e-06
beta_intersection_no_intersection_I,0.336848,0.133030,2.532127,1.133728e-02
beta_lighting_conditions_night_with_street_lightings_on_F,0.898662,0.288812,3.111579,1.860897e-03
beta_maneuver_2_overtaking_I,-0.355888,0.212654,-1.673555,9.421799e-02
beta_maneuver_2_turning_right_F,1.246840,0.313720,3.974371,7.056540e-05
beta_maneuver_without_change_of_direction_I,0.404632,0.115963,3.489335,4.842234e-04
beta_number_of_involved_vehicles_I,-0.964021,0.205117,-4.699849,2.603543e-06


### Testing the passenger restriction

`beta_user_category_passenger_I` enters both the injury and the fatality utility:
being a passenger is assumed to shift the two severities by the same amount. The
cell below re-estimates the model with a free fatality coefficient and tests
$H_0:\;\beta_I^{\text{passenger}} = \beta_F^{\text{passenger}}$ with a Wald
($t$) test on the difference, computed from the robust variance-covariance
matrix, and with the corresponding likelihood-ratio test.


In [12]:
# --- Is the passenger effect the same on injury and on fatality? --------------
# The specification above deliberately shares ONE coefficient
# (`beta_user_category_passenger_I`) between the injury and the fatality utility.
# To justify that restriction, the same model is re-estimated with a free
# fatality coefficient, and the two are compared with a Wald (t) test on their
# difference, using the robust variance-covariance matrix:
#
#     t = (b_I - b_F) / sqrt( var(b_I) + var(b_F) - 2 cov(b_I, b_F) )
#
# Not rejecting H0: b_I = b_F is what allows the constrained model to be kept.

import math


def _beta_values(results):
    for name in ('get_beta_values', 'getBetaValues'):
        method = getattr(results, name, None)
        if method is not None:
            return method()
    return dict(zip(results.data.betaNames, results.data.betaValues))


def _robust_var_covar(results):
    """Matrice de variance-covariance robuste, en DataFrame indexe par les betas."""
    matrix = None
    for name in ('get_robust_var_covar', 'getRobustVarCovar'):
        method = getattr(results, name, None)
        if method is not None:
            matrix = method()
            break
    if matrix is None:
        matrix = results.data.robust_varCovar
    if not isinstance(matrix, pd.DataFrame):
        names = list(_beta_values(results))
        matrix = pd.DataFrame(matrix, index=names, columns=names)
    return matrix


def equality_t_test(results, first, second):
    """Test de Wald H0 : beta_first = beta_second (covariance robuste).

    Retourne (difference, ecart-type, t, p). Les deux parametres doivent etre
    estimes dans le MEME modele, sinon la covariance n'est pas definie.
    """
    values = _beta_values(results)
    covariance = _robust_var_covar(results)
    difference = float(values[first]) - float(values[second])
    variance = (covariance.loc[first, first] + covariance.loc[second, second]
                - 2 * covariance.loc[first, second])
    std_error = math.sqrt(float(variance))
    t_statistic = difference / std_error
    p_value = 2 * (1 - norm.cdf(abs(t_statistic)))
    return difference, std_error, t_statistic, p_value


# Modele sans contrainte : meme specification, mais un coefficient passager
# propre a la fatalite. Ecrire la difference des deux betas evite de recopier
# toute l'utilite : le coefficient contraint s'annule et seul `..._F` subsiste.
v_fatality_free = v_fatality + (
    beta_user_category_passenger_F - beta_user_category_passenger_I
) * user_category_passenger

utility_passenger_free = {1: v_no_injury, 2: v_injury, 3: v_fatality_free}

logprob_passenger_free = models.loglogit(utility_passenger_free, availability, severity)
model_car_passenger_free = bio.BIOGEME(database_carcrashes, logprob_passenger_free)
model_car_passenger_free.modelName = 'logit_car_crashes_passenger_free'
results_car_passenger_free = estimate_in('carcrashes', model_car_passenger_free)

difference, std_error, t_statistic, p_value = equality_t_test(
    results_car_passenger_free,
    'beta_user_category_passenger_I', 'beta_user_category_passenger_F',
)

estimates = results_car_passenger_free.get_estimated_parameters()
print('Unconstrained model (one passenger coefficient per alternative):')
print(estimates.loc[['beta_user_category_passenger_I',
                     'beta_user_category_passenger_F']].round(4))
print()
print('H0: passenger effect on injury = passenger effect on fatality')
print(f'  difference b_I - b_F : {difference:8.4f}')
print(f'  robust std error     : {std_error:8.4f}')
print(f'  t-statistic          : {t_statistic:8.4f}')
print(f'  p-value              : {p_value:8.4f}')
print('  ->', 'H0 rejected at 5%: the two effects differ.' if p_value < 0.05
      else 'H0 not rejected at 5%: a single shared coefficient is enough.')
print()

# Meme conclusion par rapport de vraisemblance : le modele contraint est
# imbrique dans le modele libre (une restriction, donc 1 degre de liberte).
print('Likelihood-ratio test, constrained vs. free:')
print(results_ml_carcrashes.likelihood_ratio_test(results_car_passenger_free, 0.05))

Unconstrained model (one passenger coefficient per alternative):
                                 Value  Rob. Std err  Rob. t-test  \
beta_user_category_passenger_I -2.1913        0.2283      -9.5982   
beta_user_category_passenger_F -3.0474        0.8882      -3.4311   

                                Rob. p-value  
beta_user_category_passenger_I        0.0000  
beta_user_category_passenger_F        0.0006  

H0: passenger effect on injury = passenger effect on fatality
  difference b_I - b_F :   0.8561
  robust std error     :   0.8086
  t-statistic          :   1.0587
  p-value              :   0.2897
  -> H0 not rejected at 5%: a single shared coefficient is enough.

Likelihood-ratio test, constrained vs. free:
LRTuple(message='H0 cannot be rejected at level 5.0%', statistic=1.3892959987483664, threshold=np.float64(3.8414588206941205))


### Ordered-probit counterpart

An ordered model has a single latent index, so it cannot give a variable to one
severity level only. The index below is therefore the **union** of the
regressors of the injury and fatality utilities -- fifteen terms, against the
twenty coefficients the MNL spends on the same information.

No constant enters the index: the threshold plays that role.


In [13]:
# --- Car crashes: ordered-probit counterpart of the MNL above -----------------

car_index_terms = {
    'gender_female': gender_female,
    'age': age,
    'user_category_passenger': user_category_passenger,
    'impact_back_bike': point_of_impact_back * (vehicle_bike + vehicle_e_bike),
    'impact_back_epmd': point_of_impact_back * vehicle_e_pmd,
    'light_motorized_vehicle': vehicle_type_2_light_motorized_vehicle,
    'large_motorized_vehicle': vehicle_type_2_large_motorized_vehicle,
    'overtaking': maneuver_2_overtaking,
    'no_change_of_direction_bike': maneuver_without_change_of_direction * vehicle_bike,
    'female_driver_passenger': user_category_passenger * gender_driver_female,
    'at_intersection': (intersection_no_intersection == 0),
    'vma_no_intersection': vma * intersection_no_intersection,
    'night_street_lightings_on': lighting_conditions_night_with_street_lightings_on,
    'night_no_street_lightings': lighting_conditions_night_without_street_lightings,
    'turning_right': maneuver_2_turning_right,
    'on_cycle_facility': accident_location_on_cycle_facility,
}

index_car = None
for name, term in car_index_terms.items():
    contribution = Beta(f'beta_{name}_car_probit', 0, None, None, 0) * term
    index_car = contribution if index_car is None else index_car + contribution

model_name = 'ordered_probit_carcrashes'
the_proba = ordered_probit(
    continuous_value=index_car,
    list_of_discrete_values=[1, 2, 3],
    tau_parameter=Beta('tau_1_car_probit', 1, None, None, 0),
)
logprob = log(Elem(the_proba, severity))
model_car_probit = bio.BIOGEME(database_carcrashes, logprob)
model_car_probit.modelName = model_name
results_car_probit = estimate_in('carcrashes', model_car_probit)

print(f'MNL            : LL={results_ml_carcrashes.data.logLike:9.3f}  '
      f'K={results_ml_carcrashes.data.nparam:3d}  '
      f'AIC={results_ml_carcrashes.data.akaike:8.2f}')
print(f'ordered probit : LL={results_car_probit.data.logLike:9.3f}  '
      f'K={results_car_probit.data.nparam:3d}  '
      f'AIC={results_car_probit.data.akaike:8.2f}')
results_car_probit.get_estimated_parameters().round(4)


MNL            : LL=-1445.631  K= 17  AIC= 2925.26
ordered probit : LL=-1489.701  K= 18  AIC= 3015.40


,Value,Rob. Std err,Rob. t-test,Rob. p-value
beta_age_car_probit,0.0087,0.0019,4.6537,0.0000
beta_at_intersection_car_probit,-0.0878,0.0538,-1.6310,0.1029
beta_female_driver_passenger_car_probit,-0.4771,0.2288,-2.0855,0.0370
beta_gender_female_car_probit,0.1879,0.0521,3.6079,0.0003
beta_impact_back_bike_car_probit,-0.0352,0.0901,-0.3903,0.6963
beta_impact_back_epmd_car_probit,-0.2493,0.2036,-1.2246,0.2207
beta_large_motorized_vehicle_car_probit,0.6398,0.1611,3.9714,0.0001
beta_light_motorized_vehicle_car_probit,-1.0548,0.0601,-17.5630,0.0000
beta_night_no_street_lightings_car_probit,0.1794,0.3075,0.5832,0.5598
beta_night_street_lightings_on_car_probit,0.1040,0.0623,1.6701,0.0949


### Ordered-logit counterpart

The same latent index as the ordered probit, with a logistic error term instead
of a normal one. Comparing the two isolates the effect of the link function.


In [14]:
# --- Car crashes: ordered-logit counterpart ----------------------------------
# Exactly the latent index of the ordered probit above: only the link function
# changes, so the two rows of the comparison table differ by the distribution
# assumed for the error term and by nothing else. The coefficients get their own
# names (`_car_logit`) so that the two estimations stay independent -- a logit
# index is scaled by pi/sqrt(3) against a probit one, the values are not
# directly comparable.

index_car_logit = None
for name, term in car_index_terms.items():
    contribution = Beta(f'beta_{name}_car_logit', 0, None, None, 0) * term
    index_car_logit = (contribution if index_car_logit is None
                       else index_car_logit + contribution)

model_name = 'ordered_logit_carcrashes'
the_proba = ordered_logit(
    continuous_value=index_car_logit,
    list_of_discrete_values=[1, 2, 3],
    tau_parameter=Beta('tau_1_car_logit', 1, None, None, 0),
)
logprob = log(Elem(the_proba, severity))
model_car_logit = bio.BIOGEME(database_carcrashes, logprob)
model_car_logit.modelName = model_name
results_car_logit = estimate_in('carcrashes', model_car_logit)

for label, results in (('MNL           ', results_ml_carcrashes),
                       ('ordered probit', results_car_probit),
                       ('ordered logit ', results_car_logit)):
    print(f'{label} : LL={results.data.logLike:9.3f}  K={results.data.nparam:3d}  '
          f'AIC={results.data.akaike:8.2f}  BIC={results.data.bayesian:8.2f}')
results_car_logit.get_estimated_parameters().round(4)

MNL            : LL=-1445.631  K= 17  AIC= 2925.26  BIC= 3046.24
ordered probit : LL=-1489.701  K= 18  AIC= 3015.40  BIC= 3143.49
ordered logit  : LL=-1488.566  K= 18  AIC= 3013.13  BIC= 3141.22


,Value,Rob. Std err,Rob. t-test,Rob. p-value
beta_age_car_logit,0.0211,0.0045,4.6763,0.0000
beta_at_intersection_car_logit,-0.2330,0.1230,-1.8946,0.0581
beta_female_driver_passenger_car_logit,-0.9080,0.4349,-2.0878,0.0368
beta_gender_female_car_logit,0.4992,0.1202,4.1530,0.0000
beta_impact_back_bike_car_logit,-0.1226,0.1936,-0.6333,0.5265
beta_impact_back_epmd_car_logit,-0.6048,0.3553,-1.7023,0.0887
beta_large_motorized_vehicle_car_logit,1.7789,0.4145,4.2916,0.0000
beta_light_motorized_vehicle_car_logit,-2.2557,0.1273,-17.7243,0.0000
beta_night_no_street_lightings_car_logit,0.1708,0.7659,0.2230,0.8235
beta_night_street_lightings_on_car_logit,0.1930,0.1374,1.4050,0.1600


## MMV

### Baseline (constants only)


In [15]:
v_no_injurym=0
v_injurym = constant_I


U={1:v_no_injurym,2:v_injurym}

In [16]:
model_name = 'InitialModel_mmv'

# La disponibilite, pas les utilites : les deux alternatives sont toujours
# offertes. L'ancienne version passait {1: 0, 2: constant_I}, ce qui rendait
# l'alternative 1 indisponible -- d'ou l'avertissement "chosen alternative
# is not available" sur toutes les lignes sans blessure -- et faisait d'un
# parametre estime une disponibilite.
availability1 = {1: 1, 2: 1}

logprob_2 = models.loglogit(U, availability_harmed, harmed)

# Créez l'objet Biogeme
model_cst_mmv = bio.BIOGEME(database_mmv, logprob_2)
model_cst_mmv.modelName = model_name

# Estimation


results_constant_mmv = estimate_in('mmv', model_cst_mmv)



### Full logit specification


In [17]:
v_no_injury = 0

v_injury = (
      beta_gender_female_I                  * gender_female
    + constant_I
    + beta_gender_2_female_I               * (gender_2_female * (age_opposite_missing == 0))
    + beta_point_of_impact_back_I_bike     * point_of_impact_back * (vehicle_bike + vehicle_e_bike)
        + beta_point_of_impact_back_I_epmd     * point_of_impact_back * (vehicle_e_pmd)
+ beta_vehicle_2_e_pmd_I*any_of(vehicle_2_e_pmd, vehicle_3_e_pmd)*(age_opposite_missing==0)
    + beta_surface_condition_wet_I         * surface_condition_wet
    + beta_age_I                           * age
     + beta_maneuver_swerving_I           * maneuver_swerving
    + beta_maneuver_turning_left_I         * maneuver_turning_left
    + beta_age_2_I                          * age_opposite_mean
  #  + beta_age_opposite_missing_I           * age_opposite_missing
)

# Dictionnaire des fonctions d'utilité
utility_mmv = {
    1: v_no_injury,
    2: v_injury,
}


In [18]:
availability1={1:1,2:1}



logprob_2 = models.loglogit(utility_mmv, availability_harmed, harmed)
model_mmv = bio.BIOGEME(database_mmv, logprob_2)
model_mmv.modelName = 'logit_mmv'
results_logit_mmv = estimate_in('mmv', model_mmv)
results_logit_mmv.get_estimated_parameters()

,Value,Rob. Std err,Rob. t-test,Rob. p-value
beta_age_2_I,-0.026696,0.005030,-5.307871,1.109129e-07
beta_age_I,0.034354,0.005182,6.629825,3.360867e-11
beta_gender_2_female_I,-1.190180,0.144109,-8.258913,2.220446e-16
beta_gender_female_I,1.104489,0.165974,6.654601,2.840683e-11
beta_maneuver_swerving_I,-0.562736,0.193576,-2.907048,3.648573e-03
beta_maneuver_turning_left_I,-1.192711,0.302309,-3.945334,7.968872e-05
beta_point_of_impact_back_I_bike,-1.166689,0.215963,-5.402255,6.580829e-08
beta_point_of_impact_back_I_epmd,1.345808,0.669842,2.009142,4.452212e-02
beta_surface_condition_wet_I,-0.547489,0.229315,-2.387497,1.696355e-02
beta_vehicle_2_e_pmd_I,-0.397637,0.175783,-2.262083,2.369230e-02


### Binary probit counterpart

With two outcomes the ordered logit *is* the binary logit estimated above, so
the ordered/unordered comparison has nothing to arbitrate here: only the link
function can be varied. `ordered_probit` over two discrete values is exactly a
binary probit. Its coefficients carry the opposite sign, because the ordered
parameterisation models P(severity = 1) = Phi(tau - V), and they are on the
normal rather than the logistic scale -- what is comparable is the
log-likelihood, hence the AIC.

The constant is dropped from the index: in the ordered parameterisation the
threshold `tau` plays that role, and keeping both would not be identified.


In [19]:
# --- MMV: binary probit counterpart of the binary logit above -----------------

index_mmv = (
      Beta('beta_gender_female_mmv_probit', 0, None, None, 0) * gender_female
    + Beta('beta_gender_2_female_mmv_probit', 0, None, None, 0)
      * (gender_2_female + gender_3_male)
    + Beta('beta_impact_back_bike_mmv_probit', 0, None, None, 0)
      * point_of_impact_back * (vehicle_bike + vehicle_e_bike)
    + Beta('beta_impact_back_epmd_mmv_probit', 0, None, None, 0)
      * point_of_impact_back * vehicle_e_pmd
    + Beta('beta_surface_wet_mmv_probit', 0, None, None, 0) * surface_condition_wet
    + Beta('beta_age_mmv_probit', 0, None, None, 0) * age
    + Beta('beta_swerving_mmv_probit', 0, None, None, 0) * maneuver_swerving
    + Beta('beta_turning_left_mmv_probit', 0, None, None, 0) * maneuver_turning_left
    + Beta('beta_age_opposite_mean_mmv_probit', 0, None, None, 0) * age_opposite_mean
    + Beta('beta_age_opposite_missing_mmv_probit', 0, None, None, 0) * age_opposite_missing
    + Beta('beta_vehicle_2_e_pmd_mmv_probit', 0, None, None, 0)
      * (vehicle_2_e_pmd + vehicle_3_e_pmd)
)

model_name = 'binary_probit_mmv'
the_proba = ordered_probit(
    continuous_value=index_mmv,
    list_of_discrete_values=[1, 2],
    tau_parameter=Beta('tau_1_mmv_probit', 1, None, None, 0),
)
logprob = log(Elem(the_proba, harmed))
model_mmv_probit = bio.BIOGEME(database_mmv, logprob)
model_mmv_probit.modelName = model_name
results_mmv_probit = estimate_in('mmv', model_mmv_probit)

print(f'binary logit  : LL={results_logit_mmv.data.logLike:9.3f}  '
      f'K={results_logit_mmv.data.nparam:3d}  '
      f'AIC={results_logit_mmv.data.akaike:8.2f}')
print(f'binary probit : LL={results_mmv_probit.data.logLike:9.3f}  '
      f'K={results_mmv_probit.data.nparam:3d}  AIC={results_mmv_probit.data.akaike:8.2f}')
results_mmv_probit.get_estimated_parameters().round(4)


binary logit  : LL= -686.203  K= 11  AIC= 1394.41
binary probit : LL= -665.448  K= 12  AIC= 1354.90


,Value,Rob. Std err,Rob. t-test,Rob. p-value
beta_age_mmv_probit,0.0199,0.0030,6.6050,0.0000
beta_age_opposite_mean_mmv_probit,-0.0163,0.0029,-5.6862,0.0000
beta_age_opposite_missing_mmv_probit,3.5360,0.0978,36.1712,0.0000
beta_gender_2_female_mmv_probit,-0.5449,0.0840,-6.4850,0.0000
beta_gender_female_mmv_probit,0.6172,0.0966,6.3861,0.0000
beta_impact_back_bike_mmv_probit,-0.6526,0.1340,-4.8711,0.0000
beta_impact_back_epmd_mmv_probit,0.7263,0.4026,1.8037,0.0713
beta_surface_wet_mmv_probit,-0.3463,0.1366,-2.5350,0.0112
beta_swerving_mmv_probit,-0.3075,0.1154,-2.6658,0.0077
beta_turning_left_mmv_probit,-0.7701,0.1925,-4.0002,0.0001


## Pedestrian

### Baseline (constants only)

Ordered-probit baseline with a flat utility (`continuous_value=0`) and the
single threshold `tau_1`.


In [20]:
model_name = 'ordered_probit_pedestrian_init'

tau_1 = Beta('tau_1', 1, None, None, 0)
   
# :math:`\tau_2 = \tau_2 + \delta_2`
the_proba = ordered_probit(
    continuous_value=0,
    list_of_discrete_values=[1, 2, 3],
    tau_parameter=tau_1,
)

the_chosen_proba = Elem(the_proba, severity)

logprob = log((the_chosen_proba))
model_cst_pedes = bio.BIOGEME(database_pedestrian, logprob)
model_cst_pedes.modelName = model_name
results_pedes_cst = estimate_in('pedestrian', model_cst_pedes)


### Full ordered-probit specification


In [21]:
utility_pedestrian = (
      beta_gender_female                     * gender_female
    + beta_age                               * age
    + beta_intersection_no_intersection      * (intersection_no_intersection)
    + beta_age_opposite_mean                             * age_opposite_mean
    + beta_age_opposite_missing                          * age_opposite_missing
    + beta_gender_2_female                   * gender_2_female
    + beta_user_category_pedestrian          * user_category_pedestrian
    # « Au moins un des tiers tourne » : la somme brute donnerait 2 si le
    # premier et le second tiers tournaient tous les deux, et ignorait le cas
    # ou seul le second tiers tourne.

    + beta_crossroad_traffic_lights          * crossroad_traffic_lights
)


In [22]:
# --- Pedestrian: ordered-probit counterpart of the ordered logit above --------
# Same index (`utility_pedestrian`) and same threshold structure; only the link
# function changes. The two estimations are independent, so the Beta objects can
# be shared: each `estimate()` starts from the initial values again.

model_name = 'ordered_probit_pedestrian'

tau_1_pedes_probit = Beta('tau_1_pedes_probit', 1, None, None, 0)
the_proba = ordered_probit(
    continuous_value=utility_pedestrian,
    list_of_discrete_values=[1, 2, 3],
    tau_parameter=tau_1_pedes_probit,
)
logprob = log(Elem(the_proba, severity))
model_pedes_probit = bio.BIOGEME(database_pedestrian, logprob)
model_pedes_probit.modelName = model_name
results_pedes_probit = estimate_in('pedestrian', model_pedes_probit)


results_pedes_probit.get_estimated_parameters().round(4)

,Value,Rob. Std err,Rob. t-test,Rob. p-value
beta_age,0.0146,0.0016,9.0997,0.0000
beta_age_opposite_mean,-0.0109,0.0015,-7.2931,0.0000
beta_age_opposite_missing,1.2898,0.4885,2.6402,0.0083
beta_crossroad_traffic_lights,0.1990,0.0779,2.5547,0.0106
beta_gender_2_female,-0.5870,0.0653,-8.9957,0.0000
beta_gender_female,0.5115,0.0670,7.6321,0.0000
beta_intersection_no_intersection,0.2216,0.0694,3.1913,0.0014
beta_user_category_pedestrian,1.3222,0.0728,18.1616,0.0000
tau_1_pedes_probit,0.4926,0.0717,6.8715,0.0000
tau_1_pedes_probit_diff_2,4.1837,0.1665,25.1217,0.0000


### MNL counterpart of the same specification

The same nine variables, but one coefficient per alternative instead of a single
latent index. The ordered probit forces a variable to push severity in one
direction only; the MNL lets it raise the odds of injury and lower those of
death, at the cost of twice the coefficients.

The interaction `pedestrian x turning` never occurs among the fatalities, so its
coefficient on that alternative is not identified. The cell prints the counts.


In [23]:
# --- Pedestrian: binary logit, injury and fatality merged ---------------------
# Nine deaths in the segment cannot identify a fatality-specific utility, so the
# two harmful outcomes are merged and the model asks whether the person was
# harmed at all. The random intercept below is kept, commented out, as the mixed
# logit counterpart:
#
#     ASC_injury,n = asc_injury_ped + sigma_injury_ped * xi_n ,   xi_n ~ N(0, 1)
#
# It absorbs whatever raises or lowers the propensity to be injured and is not in
# the regressors. The choice probability is the logit probability integrated over
# `xi`, computed by Monte-Carlo simulation. `sigma_injury_ped` is the parameter
# that matters: not significantly different from zero means no unobserved
# heterogeneity to capture, and the MNL is enough.
#
# The same error component could be added to the fatality utility (a second
# `sigma`), at the cost of one more parameter and one more dimension to simulate.

from biogeme.expressions import MonteCarlo

xi_ped = bioDraws('xi_ped', 'NORMAL')
sigma_injury_ped = Beta('sigma_injury_ped', 1, None, None, 0)
beta_maneuver_2_turning = Beta('beta_maneuver_2_turning', 0, None, None, 0)
v_injury_ped_mnl = (Beta('asc_injury_ped', 0, None, None, 0)
   # + sigma_injury_ped * xi_ped
    + beta_gender_female_I                     * gender_female
    + beta_age_I                               * age
    + beta_intersection_no_intersection_I      * (intersection_no_intersection == 0)
    + beta_age_opposite_mean_I                 * age_opposite_mean
    + beta_age_opposite_missing_I              * age_opposite_missing
    + beta_gender_2_female_I                   * gender_2_female
    + beta_user_category_pedestrian_I          * user_category_pedestrian
    + beta_crossroad_traffic_lights_I          * crossroad_traffic_lights
      
)

# LA FUSION SE FAIT SUR LA VARIABLE EXPLIQUEE, PAS SUR LES UTILITES.
#
# Ecrire `{1: 0, 2: V, 3: V}` ne fusionne pas la blessure et le deces : dans un
# logit la probabilite ne depend que de l'utilite, donc deux alternatives de
# meme utilite sont EQUIPROBABLES par construction -- la constante etant elle
# aussi partagee, rien ne rattrape le niveau. Le modele predisait donc autant de
# deces que de blesses, la ou le segment en compte 1 603 contre 9. Le cout
# etait de 1 117 points de log-vraisemblance a nombre de parametres identique
# (-2 175.35 contre -1 058.00), et surtout des coefficients estimes sous une
# contrainte que les donnees contredisent.
#
# On recode donc la reponse en deux modalites -- 1 indemne, 2 blesse ou tue --
# et on n'ecrit qu'une utilite. `harmed` est une expression Biogeme, ce qui
# evite de reconstruire la base de donnees.
utility_pedestrian_mnl = {1: 0, 2: v_injury_ped_mnl}

model_name = 'logit_pedestrian'

logprob = models.loglogit(utility_pedestrian_mnl, availability_harmed, harmed)

model_pedestrian_mnl = bio.BIOGEME(database_pedestrian, logprob)
model_pedestrian_mnl.modelName = model_name
results_pedestrian_mnl = estimate_in('pedestrian', model_pedestrian_mnl)
results_pedestrian_mnl.get_estimated_parameters().round(4)

,Value,Rob. Std err,Rob. t-test,Rob. p-value
asc_injury_ped,-0.4553,0.1084,-4.2010,0.0000
beta_age_I,0.0266,0.0028,9.4131,0.0000
beta_age_opposite_mean_I,-0.0184,0.0027,-6.9146,0.0000
beta_age_opposite_missing_I,2.1109,0.8253,2.5578,0.0105
beta_crossroad_traffic_lights_I,0.3360,0.1495,2.2480,0.0246
beta_gender_2_female_I,-1.0951,0.1139,-9.6129,0.0000
beta_gender_female_I,0.9958,0.1163,8.5629,0.0000
beta_intersection_no_intersection_I,-0.3566,0.1305,-2.7324,0.0063
beta_user_category_pedestrian_I,2.2874,0.1240,18.4538,0.0000


In [24]:
# --- Pedestrian: constants-only counterpart of the binary logit ---------------
# The reference for the out-of-sample rho-square, 1 - LL(beta) / LL(c), must be
# in the SAME framework and on the SAME dependent variable as the model it is
# compared with. `model_cst_pedes` above is an ordered probit over three
# severity levels, whereas the model retained here is a binary logit on
# `harmed`: their LL(c) are not comparable, a null model over three categories
# being far below a null model over two. Hence this one.

model_name = 'logit_pedestrian_cst'

logprob = models.loglogit(
    {1: 0, 2: Beta('asc_harmed_ped_cst', 0, None, None, 0)},
    availability_harmed,
    harmed,
)
model_cst_pedes_mnl = bio.BIOGEME(database_pedestrian, logprob)
model_cst_pedes_mnl.modelName = model_name
results_pedes_cst_mnl = estimate_in('pedestrian', model_cst_pedes_mnl)

print(f'LL(c) binary logit  : {results_pedes_cst_mnl.data.logLike:10.3f}  '
      f'(2 modalites de `harmed`)')
print(f'LL(c) ordered probit: {results_pedes_cst.data.logLike:10.3f}  '
      f'(3 niveaux de `severity`)')
results_pedes_cst_mnl.get_estimated_parameters().round(4)


LL(c) binary logit  :  -1901.711  (2 modalites de `harmed`)
LL(c) ordered probit:  -1957.378  (3 niveaux de `severity`)


,Value,Rob. Std err,Rob. t-test,Rob. p-value
asc_harmed_ped_cst,0.312,0.0383,8.1426,0.0


## Single-vehicle

### Baseline (constants only)

Ordered-probit baseline (`continuous_value=0`).


In [25]:
model_name = 'ordered_probit_s_init'

tau_1 = Beta('tau_1', 1, None, None, 0)
   
# :math:`\tau_2 = \tau_2 + \delta_2`
the_proba = ordered_probit(
    continuous_value=0,
    list_of_discrete_values=[1, 2, 3],
    tau_parameter=tau_1,
)

the_chosen_proba = Elem(the_proba, severity)

logprob = log((the_chosen_proba))
model_cst_solo = bio.BIOGEME(database_sv, logprob)
model_cst_solo.modelName = model_name
results_cst_solo = estimate_in('single_vehicle', model_cst_solo)
results_cst_solo.get_estimated_parameters()

,Value,Rob. Std err,Rob. t-test,Rob. p-value
tau_1,-2.115844,0.064947,-32.578097,0.0
tau_1_diff_2,4.395907,0.099121,44.348801,0.0


### Full ordered-probit specification


In [26]:
utility_sv = (
   beta_age * age +
    beta_user_category_passenger * user_category_passenger +
    beta_long_profile_slope *long_profile_slope 
     + beta_helmet_yes_ebike*helmet_yes*(vehicle_e_bike)
  #  + beta_number_of_passengers*user_category_driver*(number_of_passengers>0)
)


In [27]:
model_name = 'ordered_probit_sinv'

tau_1 = Beta('tau_1', 1, None, None, 0)
   
# :math:`\tau_2 = \tau_2 + \delta_2`
the_proba = ordered_probit(
    continuous_value=utility_sv,
    list_of_discrete_values=[1, 2, 3],
    tau_parameter=tau_1,
)

the_chosen_proba = Elem(the_proba, severity)

logprob = log((the_chosen_proba))
model_solo_2 = bio.BIOGEME(database_sv, logprob)
model_solo_2.modelName = model_name
results_solo_2 = estimate_in('single_vehicle', model_solo_2)
results_solo_2.get_estimated_parameters()

,Value,Rob. Std err,Rob. t-test,Rob. p-value
beta_age,0.011141,0.003141,3.547078,3.895289e-04
beta_helmet_yes_ebike,-0.186252,0.128279,-1.451929,1.465212e-01
beta_long_profile_slope,0.303972,0.152742,1.990099,4.657998e-02
beta_user_category_passenger,-1.359850,0.227463,-5.978341,2.254219e-09
tau_1,-2.290105,0.086301,-26.536260,0.000000e+00
tau_1_diff_2,4.672433,0.123742,37.759469,0.000000e+00


### Ordered-logit counterpart

The same index with a logistic error term.


In [28]:
# --- Single-vehicle: ordered-logit counterpart of the ordered probit above ----
# Same index (`utility_sv`), same thresholds; only the link function changes.

model_name = 'ordered_logit_sinv'

tau_1_solo_logit = Beta('tau_1_solo_logit', 1, None, None, 0)
the_proba = ordered_logit(
    continuous_value=utility_sv,
    list_of_discrete_values=[1, 2, 3],
    tau_parameter=tau_1_solo_logit,
)
logprob = log(Elem(the_proba, severity))
model_solo_logit = bio.BIOGEME(database_sv, logprob)
model_solo_logit.modelName = model_name
results_solo_logit = estimate_in('single_vehicle', model_solo_logit)

for label, results in (('ordered probit', results_solo_2),
                       ('ordered logit ', results_solo_logit)):
    print(f'{label} : LL={results.data.logLike:9.3f}  K={results.data.nparam:3d}  '
          f'AIC={results.data.akaike:8.2f}  BIC={results.data.bayesian:8.2f}')
results_solo_logit.get_estimated_parameters().round(4)

ordered probit : LL= -289.842  K=  6  AIC=  591.68  BIC=  625.89
ordered logit  : LL= -288.703  K=  6  AIC=  589.41  BIC=  623.62


,Value,Rob. Std err,Rob. t-test,Rob. p-value
beta_age,0.0272,0.0078,3.5051,0.0005
beta_helmet_yes_ebike,-0.4593,0.3970,-1.1568,0.2473
beta_long_profile_slope,0.7004,0.3693,1.8966,0.0579
beta_user_category_passenger,-2.9644,0.4072,-7.2805,0.0000
tau_1_solo_logit,-4.5349,0.2260,-20.0684,0.0000
tau_1_solo_logit_diff_2,9.2663,0.3185,29.0945,0.0000


### MNL counterpart of the same specification

The same five variables, but one coefficient per alternative instead of a single
latent index. The ordered probit constrains every variable to push severity in
one direction; the MNL lets a variable raise the odds of being injured while
lowering those of being killed. That freedom costs twice the coefficients, which
is what the framework comparison further down arbitrates.

Two of these terms are perfectly separated from the fatality outcome, so their
`_F` coefficients are not identified and will run off towards minus infinity.
The cell prints the counts that cause it; the estimates it reports for those two
coefficients are artefacts, not effects.


In [29]:
# --- Single-vehicle: MNL counterpart of the ordered probit above --------------
beta_helmet_yes_ebike_I = Beta('beta_helmet_yes_ebike_I', 0, None, None, 0)
beta_helmet_yes_ebike_F = Beta('beta_helmet_yes_ebike_F', 0, None, None, 0)

v_injury_sv = (Beta('asc_injury_sv', 0, None, None, 0)
    + beta_age_I * age
    + beta_user_category_passenger_I * user_category_passenger
    + beta_vehicle_e_pmd*vehicle_e_pmd

   # + beta_number_of_passengers_I * user_category_driver * number_of_passengers  # ajouté
)
v_fatality_sv = (Beta('asc_fatality_sv', 0, None, None, 0)
 +   beta_age_F * age +
    beta_long_profile_slope_F *long_profile_slope 
)


utility_sv_mnl = {1: 0, 2: v_injury_sv, 3: v_fatality_sv}

# Which coefficients cannot be identified on the fatality alternative.
print('Counts by severity (1 / 2 / 3) for the two interaction terms:')
for label, column in (('helmet_ebike', (df_sv['Helmet_Yes'] * df_sv['Vehicle_E-bike']) > 0),
                      ('passengers_driver',
                       (df_sv['User category_Driver'] * df_sv['Number of passengers']) > 0)):
    counts = df_sv.loc[column, 'severity'].value_counts().reindex([1, 2, 3], fill_value=0)
    print(f'  {label:18s} = 1 -> {counts[1]:4d} / {counts[2]:4d} / {counts[3]:4d}')
print()

model_name = 'mnl_sinv'
logprob = models.loglogit(utility_sv_mnl, availability, severity)
model_solo_mnl = bio.BIOGEME(database_sv, logprob)
model_solo_mnl.modelName = model_name
results_solo_mnl = estimate_in('single_vehicle', model_solo_mnl)



Counts by severity (1 / 2 / 3) for the two interaction terms:
  helmet_ebike       = 1 ->    1 /   88 /    0
  passengers_driver  = 1 ->   19 /   46 /    0



In [30]:
# --- Single-vehicle: constants-only counterpart of the MNL --------------------
# Three alternatives, as in `utility_sv_mnl`: one alternative-specific constant
# per harmful outcome, the reference being normalised to zero. With constants
# alone the model reproduces the observed severity shares exactly, so this
# LL(c) equals that of the ordered-probit baseline above -- the two are written
# separately only so that each model is referenced against its own framework.

model_name = 'mnl_sinv_cst'

logprob = models.loglogit(
    {1: 0,
     2: Beta('asc_injury_sv_cst', 0, None, None, 0),
     3: Beta('asc_fatality_sv_cst', 0, None, None, 0)},
    availability,
    severity,
)
model_cst_solo_mnl = bio.BIOGEME(database_sv, logprob)
model_cst_solo_mnl.modelName = model_name
results_cst_solo_mnl = estimate_in('single_vehicle', model_cst_solo_mnl)

print(f'LL(c) MNL           : {results_cst_solo_mnl.data.logLike:10.3f}')
print(f'LL(c) ordered probit: {results_cst_solo.data.logLike:10.3f}')
results_cst_solo_mnl.get_estimated_parameters().round(4)


LL(c) MNL           :   -328.598
LL(c) ordered probit:   -328.598


,Value,Rob. Std err,Rob. t-test,Rob. p-value
asc_fatality_sv_cst,-0.4189,0.2575,-1.6269,0.1038
asc_injury_sv_cst,4.0350,0.1636,24.6610,0.0000


In [31]:
results_solo_mnl.get_estimated_parameters().round(4)

,Value,Rob. Std err,Rob. t-test,Rob. p-value
asc_fatality_sv,-0.5720,0.3214,-1.7797,0.0751
asc_injury_sv,4.8368,0.2450,19.7420,0.0000
beta_age_F,0.0705,0.0147,4.7845,0.0000
beta_age_I,0.0267,0.0127,2.0981,0.0359
beta_long_profile_slope_F,1.0120,0.4496,2.2510,0.0244
beta_user_category_passenger_I,-2.6406,0.3609,-7.3170,0.0000
beta_vehicle_e_pmd,-0.7133,0.2824,-2.5260,0.0115


## AIC-based selection of the coefficients

Which terms would a selection driven by the AIC keep, rather than by the 5%
$t$-test? Dropping a coefficient saves one parameter and costs a
likelihood-ratio $\approx t^2$, so

$$\text{AIC}_{\text{without}} - \text{AIC}_{\text{with}} \;\approx\; t^2 - 2 ,$$

and the AIC keeps a coefficient as soon as $|t| > \sqrt{2} \approx 1.414$
($p < 0.157$). The tables below list, for the main model of each segment, the
coefficients ranked by that gain, and flag the ones the AIC keeps but the
$t$-test rejects.

Each coefficient is also dropped for real: the model is re-estimated with that
coefficient fixed at zero, which gives the exact likelihood-ratio
$LR = 2(\ell_{\text{full}} - \ell_{\text{without}})$, its $p$-value with one
degree of freedom, and the exact AIC gap. Set `EXACT_LR = False` to skip those
re-estimations while iterating.


In [32]:
# --- Which coefficients survive an AIC criterion? -----------------------------
# Dropping one coefficient costs one parameter and some log-likelihood. Writing
# the likelihood-ratio of that restriction with its Wald approximation (LR ~ t^2):
#
#     AIC(without) - AIC(with) = 2(K-1) - 2 LL_r - [2K - 2 LL_f] = LR - 2 ~ t^2 - 2
#
# So AIC keeps a coefficient as soon as |t| > sqrt(2) = 1.414, i.e. p < 0.157 --
# a far more permissive rule than the 5% t-test, which is why the MNL keeps
# variables a significance-based selection would have removed. A positive
# `Delta AIC (drop)` means the model gets WORSE without the coefficient.
#
# `Delta AIC (drop)` is that first-order approximation. Setting `EXACT_LR = True`
# adds the exact figures: the model is re-estimated once per coefficient, with
# that coefficient fixed at zero, which gives the true likelihood-ratio
# LR = 2 (LL_full - LL_without), its p-value (1 d.f.) and the exact AIC gap.
# Exact means K estimations per model, so it is slow -- switch it off to iterate.

import math
from contextlib import contextmanager

from scipy.stats import chi2
from biogeme.expressions import Beta

EXACT_LR = True
AIC_THRESHOLD = math.sqrt(2)      # |t| au-dela duquel l'AIC garde le coefficient
SIGNIFICANCE_COLUMN = 'Keep (t-test 5%)'


def _parameter_columns(table):
    """Localise value / std err / t-test / p-value, versions robustes d'abord."""
    columns = list(table.columns)

    def find(keyword):
        matches = [c for c in columns if keyword in c.lower()]
        robust = [c for c in matches if 'rob' in c.lower()]
        return (robust or matches or [None])[0]

    value = next((c for c in columns if c.lower().strip() == 'value'), None)
    return {'value': value or columns[0], 'std': find('std err'),
            't': find('t-test'), 'p': find('p-value')}


# Les seuils d'un modele ordonne ne sont pas des variables : les fixer a zero ne
# revient pas a retirer un regresseur mais a casser la structure du modele
# (tau_1 = 0 deplace toute l'echelle, et annuler l'ecart tau_1_diff_2 confond les
# deux seuils, ce qui rend la vraisemblance degeneree -- d'ou les gradients NaN).
# Ils sont donc exclus de la selection.
THRESHOLD_PATTERN = re.compile(r'^(tau|delta)', re.IGNORECASE)





def _all_betas(expression):
    """Tous les objets Beta de l'arbre de l'expression."""
    found, stack, seen = [], [expression], set()
    while stack:
        node = stack.pop()
        if id(node) in seen:
            continue
        seen.add(id(node))
        if isinstance(node, Beta):
            found.append(node)
        stack.extend(node.get_children())
    return found


@contextmanager
def _restricted_formula(expression, name, beta_values=None):

    betas = _all_betas(expression)
    targets = [beta for beta in betas if beta.name == name]
    if not targets:
        raise KeyError(f'{name} absent de la formule du modele')

    saved = [(beta, beta.status, beta.initValue) for beta in betas]
    if beta_values:
        for beta in betas:
            if beta.name in beta_values:
                beta.initValue = float(beta_values[beta.name])
    for beta in targets:
        beta.status = 1
        beta.initValue = 0.0
    try:
        yield
    finally:
        for beta, status, value in saved:
            beta.status = status
            beta.initValue = value


def lr_drop_test(the_biogeme, segment, results_full, parameter):
    """Reestime le modele avec `parameter` fixe a 0 ; retourne (LR, p, dAIC)."""
    formula = the_biogeme.log_like
    with _restricted_formula(formula, parameter, results_full.get_beta_values()):
        # L'objet Biogeme lit la formule a la construction : il doit etre
        # construit ET estime tant que le Beta est fixe.
        restricted = bio.BIOGEME(the_biogeme.database, formula,
                                 number_of_draws=the_biogeme.number_of_draws,
                                 generate_html=False, generate_pickle=False)
        restricted.modelName = f'{the_biogeme.modelName}_without_{parameter}'
        results_restricted = estimate_in(segment, restricted)

    statistic = 2 * (float(results_full.data.logLike)
                     - float(results_restricted.data.logLike))
    return (statistic, float(chi2.sf(statistic, 1)),
            float(results_restricted.data.akaike) - float(results_full.data.akaike))


def aic_selection_table(results, the_biogeme=None, segment=None, alpha=0.05):
    """Pour chaque coefficient : garde-t-on le terme sous AIC, sous t-test ?

    Si `the_biogeme` et `segment` sont fournis et `EXACT_LR` est vrai, les
    colonnes exactes (LR, p, Delta AIC) sont ajoutees, au prix d'une
    reestimation par coefficient.
    """
    table = results.get_estimated_parameters()
    column = _parameter_columns(table)

    thresholds = [name for name in table.index if THRESHOLD_PATTERN.match(str(name))]
    if thresholds:
        print(f'  seuils exclus de la selection : {", ".join(thresholds)}')
        table = table.drop(index=thresholds)

    t_statistic = table[column['t']].astype(float)
    p_value = table[column['p']].astype(float)
    delta_aic = t_statistic ** 2 - 2

    selection = pd.DataFrame({
        'Value': table[column['value']].astype(float),
        't-test': t_statistic,
        'p-value': p_value,
        'Delta AIC (drop)': delta_aic,
        'Keep (AIC)': delta_aic > 0,
        SIGNIFICANCE_COLUMN: p_value < alpha,
    })

    if EXACT_LR and the_biogeme is not None and segment is not None:
        exact = {}
        for parameter in table.index:
            try:
                exact[parameter] = lr_drop_test(the_biogeme, segment,
                                                results, parameter)
            except Exception as error:
                print(f'  [{parameter}] LR test impossible : '
                      f'{type(error).__name__}: {error}')
                exact[parameter] = (float('nan'),) * 3
        exact = pd.DataFrame(exact, index=['LR', 'p (LR)',
                                           'Delta AIC (exact)']).T
        selection = selection.join(exact)
        selection['Keep (AIC, exact)'] = selection['Delta AIC (exact)'] > 0

    return selection.sort_values('Delta AIC (drop)', ascending=False)


# Le modele « principal » de chaque segment : celui que le tableau comparatif
# designe par l'AIC. (resultats, objet Biogeme, dossier de sortie).
MAIN_MODELS = {
    'Car crashes': (results_ml_carcrashes, model_car, 'carcrashes'),
    'MMV': (results_logit_mmv, model_mmv, 'mmv'),
    'Pedestrian': (results_pedestrian_mnl, model_pedestrian_mnl, 'pedestrian'),
    'Single-vehicle': (results_solo_mnl, model_solo_mnl, 'single_vehicle'),
}

aic_selection = {}
for label, (results, the_biogeme, segment) in MAIN_MODELS.items():
    print(f'=== {label} ===')
    table = aic_selection_table(results, the_biogeme, segment)
    aic_selection[label] = table

    keep_column = 'Keep (AIC, exact)' if 'Keep (AIC, exact)' in table else 'Keep (AIC)'
    kept = table.index[table[keep_column]].tolist()
    dropped = table.index[~table[keep_column]].tolist()
    only_aic = table.index[table[keep_column] & ~table[SIGNIFICANCE_COLUMN]].tolist()

    print(f'{len(kept)}/{len(table)} coefficients kept by AIC ({keep_column})')
    print(table.round(4))
    print(f'  dropped by AIC (|t| < {AIC_THRESHOLD:.3f}) : '
          f'{", ".join(dropped) if dropped else "none"}')
    print('  kept by AIC but not significant at 5% : '
          f'{", ".join(only_aic) if only_aic else "none"}')
    print()

=== Car crashes ===
17/17 coefficients kept by AIC (Keep (AIC, exact))
                                                     Value   t-test  p-value  \
constant_I                                          3.9017  35.0498   0.0000   
beta_vehicle_type_2_light_motorized_vehicle_I      -2.0836 -17.2386   0.0000   
beta_vehicle_type_2_large_motorized_vehicle_F       3.3275  12.0862   0.0000   
constant_F                                         -2.6481 -10.6123   0.0000   
beta_user_category_passenger_I                     -2.1307  -9.6485   0.0000   
beta_age_F                                          0.0546   5.9053   0.0000   
beta_gender_female_I                                0.6392   4.7164   0.0000   
beta_number_of_involved_vehicles_I                 -0.9640  -4.6998   0.0000   
beta_maneuver_2_turning_right_F                     1.2468   3.9744   0.0001   
beta_maneuver_without_change_of_direction_I         0.4046   3.4893   0.0005   
beta_age_I                                       

In [33]:
# --- The same tables, in LaTeX ------------------------------------------------
# One table per segment, same style as the framework comparison
# (`results/model_framework_comparison.tex`): a `table` float, `\hline` rules,
# the coefficients ranked by what dropping them costs. The last two columns show
# where the two criteria disagree -- a coefficient kept by the AIC but rejected
# by the 5% t-test is exactly what justifies keeping the full specification.

from latex_tables import colorize

import re

AIC_TABLES_DIRECTORY = RESULTS_ROOT / 'tables'

# Surcharges de libelles, ex. {'beta_age_I': 'Age (years)'}.
PARAMETER_LABELS = {}

ALTERNATIVE_LABELS = {'_I': 'injury', '_F': 'fatality'}


def _escape(text):
    for char in ('&', '%', '$', '#', '_', '{', '}'):
        text = text.replace(char, '\\' + char)
    return text


def _row(cells):
    return ' & '.join(cells) + r' \\'


def _pretty(name):
    """`beta_point_of_impact_back_F` -> `Point of impact back (fatality)`."""
    if name in PARAMETER_LABELS:
        return PARAMETER_LABELS[name]
    label = str(name)
    alternative = next((text for suffix, text in ALTERNATIVE_LABELS.items()
                        if label.endswith(suffix)), None)
    label = re.sub(r'^(beta|asc)[_ ]', '', label)
    label = re.sub(r'[_ ](I|F)$', '', label)
    label = label.replace('_', ' ').strip()
    label = label[:1].upper() + label[1:]
    return f'{label} ({alternative})' if alternative else label


def _fmt_p(value):
    if value != value:                      # NaN
        return ''
    return '$<$0.001' if float(value) < 0.001 else f'{float(value):.3f}'


def make_aic_selection_table(selection, caption=None, label=None, alpha=0.05):
    """Tableau LaTeX d'une table de selection produite par `aic_selection_table`.

    L'estimation, son t, ce que couterait a l'AIC la suppression du coefficient,
    et la p-value du test du rapport de vraisemblance correspondant. Cette
    derniere n'apparait que si les colonnes exactes ont ete calculees
    (`EXACT_LR = True`) : sans re-estimation il n'y a pas de vraie p-value a
    montrer, seulement l'approximation de Wald deja resumee par le t. Le LR
    lui-meme et les deux drapeaux de decision restent dans
    `aic_selection[segment]`.
    """
    exact = 'Delta AIC (exact)' in selection.columns
    delta_column = 'Delta AIC (exact)' if exact else 'Delta AIC (drop)'
    delta_header = r'$\Delta$AIC' if exact else r'$\Delta$AIC$^{\dagger}$'

    columns = ['Parameter', 'Estimate', '$t$', delta_header]
    if exact:
        columns.append('$p$ (LR)')
    alignment = 'l' + ' r' * (len(columns) - 1)

    lines = [
        r'\begin{table}[H]',
        r'\centering',
        r'\small',
        rf'\caption{{{caption}}}' if caption else r'\caption{}',
        rf'\label{{{label}}}' if label else '',
        rf'\begin{{tabular}}{{{alignment}}}',
        r'\hline',
        _row(columns),
        r'\hline',
    ]

    for parameter, record in selection.iterrows():
        value = record[delta_column]
        cells = [_escape(_pretty(parameter)),
                 f"{float(record['Value']):.3f}",
                 f"{float(record['t-test']):.2f}",
                 '' if value != value else f'{float(value):.2f}']
        if exact:
            cells.append(_fmt_p(record['p (LR)']))
        lines.append(_row(cells))

    # La note est une LIGNE du tableau, pas un paragraphe apres `\end{tabular}` :
    # apres le tabular elle herite du `\centering` du float et se recentre, et
    # sa position depend de la classe de document. En `\multicolumn` elle reste
    # collee sous la derniere regle, alignee a gauche et a la largeur du tableau.
    note = r'\footnotesize $\Delta$AIC is AIC(without) $-$ AIC(with).'
    if exact:
        note += (r' $p$ (LR) is the exact likelihood-ratio test of dropping that '
                 r'single coefficient, the model being re-estimated without it. '
                 r'It answers a different question from $\Delta$AIC: the AIC '
                 r'keeps a coefficient as soon as it pays for itself, which '
                 r'happens around $|t| > \sqrt{2}$, whereas the LR test asks '
                 r'whether it is significant at a conventional level.')
    if not exact:
        note += (r' $^{\dagger}$ Wald approximation $t^2 - 2$; set '
                 r'\texttt{EXACT\_LR = True} for the re-estimated figures.')
    lines += [
        r'\hline',
        (rf'\multicolumn{{{len(columns)}}}'
         r'{p{\dimexpr\linewidth-2\tabcolsep}}{' + note + r'} \\'),
        r'\end{tabular}',
        r'\end{table}',
    ]
    return '\n'.join(line for line in lines if line)


AIC_TABLES_DIRECTORY.mkdir(parents=True, exist_ok=True)
aic_selection_tex = {}
for label, selection in aic_selection.items():
    stem = re.sub(r'[^a-z0-9]+', '_', label.lower()).strip('_')
    tex = make_aic_selection_table(
        selection,
        caption=(f'{label}: contribution of each coefficient to the AIC. '),
        label=f'tab:aic_selection_{stem}',
    )
    (AIC_TABLES_DIRECTORY / f'aic_selection_{stem}.tex').write_text(
        colorize(tex) + '\n', encoding='utf-8')
    aic_selection_tex[label] = tex
    print(f'{AIC_TABLES_DIRECTORY / f"aic_selection_{stem}.tex"}')

print()
print(next(iter(aic_selection_tex.values())))

results/tables/aic_selection_car_crashes.tex
results/tables/aic_selection_mmv.tex
results/tables/aic_selection_pedestrian.tex
results/tables/aic_selection_single_vehicle.tex

\begin{table}[H]
\centering
\small
\caption{Car crashes: contribution of each coefficient to the AIC. }
\label{tab:aic_selection_car_crashes}
\begin{tabular}{l r r r r}
\hline
Parameter & Estimate & $t$ & $\Delta$AIC & $p$ (LR) \\
\hline
Constant (injury) & 3.902 & 35.05 & 1393.27 & $<$0.001 \\
Vehicle type 2 light motorized vehicle (injury) & -2.084 & -17.24 & 290.61 & $<$0.001 \\
Vehicle type 2 large motorized vehicle (fatality) & 3.328 & 12.09 & 107.55 & $<$0.001 \\
Constant (fatality) & -2.648 & -10.61 & 202.70 & $<$0.001 \\
User category passenger (injury) & -2.131 & -9.65 & 88.45 & $<$0.001 \\
Age (fatality) & 0.055 & 5.91 & 40.93 & $<$0.001 \\
Gender female (injury) & 0.639 & 4.72 & 23.48 & $<$0.001 \\
Number of involved vehicles (injury) & -0.964 & -4.70 & 18.69 & $<$0.001 \\
Maneuver 2 turning right (fata

## Marginal effects

A coefficient of an MNL or of an ordered model is not an effect. What can be read
is the average marginal effect: how much $P(\text{severity} = j)$ moves when a
variable moves, averaged over the sample -- from 0 to 1 for a dummy, per unit for
a continuous variable, in percentage points. They are obtained by simulation, so
the same code covers every framework and the interactions are handled properly.

The tables are limited to the key variables of each segment (`KEY_VARIABLES`,
one list per segment) and report a 95\% Krinsky--Robb confidence interval:
parameter vectors are drawn from $N(\hat{\beta}, \Sigma)$, the whole effect is
recomputed for each draw, and the interval is read off the percentiles.


In [34]:
# --- Marginal effects ---------------------------------------------------------
# A coefficient of an MNL or of an ordered model is not an effect: its sign is
# readable, its magnitude is not, and in the ordered models a single coefficient
# moves the three probabilities at once. What can be read is the average
# marginal effect -- how much P(severity = j) moves when the variable moves,
# averaged over the sample:
#
#   dummy      : P(x = 1) - P(x = 0), averaged over the observations
#   count      : P(x + 1) - P(x), averaged -- a real one-unit change
#   continuous : [P(x + h) - P(x - h)] / (2h), averaged, i.e. a per-unit effect
#
# The distinction between count and continuous matters. A derivative is a LOCAL
# slope extrapolated over a whole unit: on `Number of passengers`, which is 0 for
# 94% of the single-vehicle sample and carries a coefficient of -2.3, that
# extrapolation returned -6.8 points of fatality probability against a baseline
# of 1.15 -- arithmetically reproducible, but meaningless as an effect. The
# discrete change on the same model gives -1.15 points, which is bounded by
# construction: the mean probability after the change stays at 0.
#
# Everything is computed by simulation rather than by a closed-form formula, so
# the same code serves the MNL, the ordered models and the mixed logit, and the
# interactions are handled correctly: perturbing `age` moves every term where
# `age` appears. Results are in PERCENTAGE POINTS.
#
# The trick for the probabilities: the log-likelihood of any of these models is
# log P(severity = observed), so forcing the severity column to j and simulating
# exp(log_like) returns P(severity = j) for every row.
#
# Point estimates only. Setting `KRINSKY_ROBB_DRAWS` to a few hundred adds a
# confidence interval: parameter vectors are then drawn from
# N(beta_hat, robust covariance), the whole effect is recomputed for each draw,
# and the interval is read off the percentiles. It costs one simulation per draw
# and per variable, hence the default of 0.

from biogeme.expressions import Variable, exp

# Pas de la difference centree, en fraction de l'ecart-type de la variable.
DERIVATIVE_STEP = 0.01
# Au-dela de ce nombre de valeurs distinctes, la variable est traitee comme
# continue (derivee) ; en deca, comme un comptage (variation discrete de +1).
DISCRETE_MAX_LEVELS = 10
# Tirages de Krinsky-Robb. 200 suffit pour deux decimales ; monter a 1000 pour
# la version finale si le temps de calcul le permet.
KRINSKY_ROBB_DRAWS = 0
CONFIDENCE_LEVEL = 0.95
# Ligne de reference : probabilite moyenne predite de chaque modalite, sans
# aucune perturbation. Elle donne l'echelle des effets -- deux points de
# pourcentage ne se lisent pas pareil sur une modalite a 0,7 % et sur une a 77 %.
BASELINE_LABEL = 'Baseline predicted probabilities'


# --- Readable names -----------------------------------------------------------
# Les indicatrices sont nommees `<colonne d'origine>_<modalite>` par
# `pd.get_dummies`. On retrouve la colonne d'origine par son prefixe, ce qui
# donne « Lighting conditions: night with street lightings on » plutot que
# « Lighting conditions night with street lightings on ».
DUMMY_PREFIXES = [
    'Crossroad', 'Helmet_driver', 'Helmet', 'Point of impact_2',
    'Point of impact_3', 'Point of impact', 'Gender_driver', 'Gender_2',
    'Gender_3', 'Gender', 'vehicle_type_2', 'vehicle_type_3', 'Vehicle_2',
    'Vehicle_3', 'Vehicle', 'Pavement', 'Intersection', 'User category',
    'Lighting conditions', 'Cycle facilities', 'Road width', 'Agglomeration',
    'Age category', 'Accident location', 'Surface condition', 'Maneuver_2',
    'Maneuver_3', 'Maneuver', 'Pedestrian localisation', 'Pedestrian action',
    'Max speed', 'Long profile', 'Weather conditions', 'Road type',
    'Trip purpose', 'Reflective jacket', 'plan', 'Obstacle', 'age_driver',
    'Year',
]

# Libelles imposes : continues, variables construites, ou tournures plus claires.
VARIABLE_LABELS = {
    'age': 'Age of the rider (years)',
    'age_2': "Age of the third party's driver (years)",
    'age_opposite_mean': "Age of the second-party's driver (years)",
    'age_opposite_missing': 'Second party not identified',
    'vma': 'Posted speed limit (km/h)',
    'number of involved vehicles': 'Number of vehicles involved',
    'Number of passengers': 'Number of passengers',
    'long_profile_slope': 'Slope',
    'Long profile_Slope': 'Slope',
    'Intersection_No intersection': 'Away from an intersection',
    'User category_Passenger': 'Passenger (vs. driver)',
    'User category_Pedestrian': 'Pedestrian',
    'Crossroad_Traffic lights': 'Signalized crossroad',
    'Accident location_On cycle facility': 'Presence of a cycle facility',
}


def _pretty_variable(name):
    """Nom lisible d'une colonne du modele."""
    if name in VARIABLE_LABELS:
        return VARIABLE_LABELS[name]
    for prefix in DUMMY_PREFIXES:
        if name.startswith(f'{prefix}_'):
            category = name[len(prefix) + 1:].replace('_', ' ')
            variable = prefix.replace('_', ' ').replace('vehicle type 2',
                                                        'Third-party vehicle')
            variable = variable[:1].upper() + variable[1:]
            return f'{variable}: {category.lower()}'
    label = name.replace('_', ' ').strip()
    return label[:1].upper() + label[1:]


# --- Key variables, per segment -----------------------------------------------
# Les variables portees dans le manuscrit. Une variable absente du modele est
# ignoree avec un avertissement ; mettre None pour reprendre toutes celles du
# modele.
# Une entree est soit un nom de colonne, soit un couple (libelle, colonnes) quand
# le modele groupe plusieurs indicatrices en un seul terme : le groupe est alors
# perturbe d'un bloc (toutes a 0 contre la premiere a 1), sinon on ne mesurerait
# qu'un morceau de l'effet -- perturber `vehicle_type_2` seul laisserait de cote
# les collisions ou le vehicule leger est le second tiers.
# Nom affiche d'un segment, quand il differe de la cle interne. La cle sert au
# nom de fichier et au `\label`, qui ne doivent pas bouger : le manuscrit y
# renvoie deja. Seuls la legende et l'intitule de bloc utilisent ce dictionnaire.
SEGMENT_DISPLAY_NAMES = {
    'Car crashes': 'Motorized vehicles',
}


def display_name(segment):
    """Libelle affiche d'un segment."""
    return SEGMENT_DISPLAY_NAMES.get(segment, segment)


KEY_VARIABLES = {
    'Car crashes': [
        'age', 'number of involved vehicles',
        ('Second-party heavy vehicle',
         ['vehicle_type_2_Large motorized vehicle',
          'vehicle_type_3_Large motorized vehicle']),
        ('Second-party light vehicle',
         ['vehicle_type_2_Light motorized vehicle',
          'vehicle_type_3_Light motorized vehicle']),
        ('Crash at night',
         ['Lighting conditions_Night with street lightings on',
          'Lighting conditions_Night without street lightings']),
        'Accident location_On cycle facility',
        'Intersection_No intersection',
        ('Second-party turning right',
         ['Maneuver_2_Turning right', 'Maneuver_3_Turning right']),
        'User category_Passenger', 'Gender_Female',
    ],
    'MMV': [
        'age', 'age_opposite_mean', 'Gender_Female', 'Surface condition_Wet',
        'Maneuver_Swerving', 'Maneuver_Turning left', 'Point of impact_Back',
    ],
    'Pedestrian': [
        'age', 'age_opposite_mean', 'Gender_Female', 'User category_Pedestrian',
        'Crossroad_Traffic lights', 'Intersection_No intersection',
    ],
    'Single-vehicle': [
        'age', 'User category_Passenger', 'Long profile_Slope',
        'Number of passengers',
    ],
}


def _model_variables(the_biogeme, exclude=('severity',)):
    """Noms des colonnes qui apparaissent dans la formule du modele."""
    found, stack, seen = set(), [the_biogeme.log_like], set()
    while stack:
        node = stack.pop()
        if id(node) in seen:
            continue
        seen.add(id(node))
        if isinstance(node, Variable):
            found.add(node.name)
        stack.extend(node.get_children())
    return sorted(found - set(exclude))


def _outcome_simulators(the_biogeme, data, outcomes, choice_column='severity'):
    """Un objet Biogeme par modalite, pretes a simuler P(severity = j)."""
    simulators = {}
    for outcome in outcomes:
        frame = data.copy()
        frame[choice_column] = outcome
        simulator = bio.BIOGEME(db.Database('marginal_effects', frame),
                                {'probability': exp(the_biogeme.log_like)},
                                number_of_draws=the_biogeme.number_of_draws,
                                generate_html=False, generate_pickle=False)
        simulator.modelName = f'{the_biogeme.modelName}_simulation'
        simulators[outcome] = simulator
    return simulators


def _beta_draws(results, count, seed=0):
    """Tirages de Krinsky-Robb dans N(beta_hat, covariance robuste)."""
    values = results.get_beta_values()
    covariance = results.get_robust_var_covar()
    names = [name for name in covariance.index if name in values]
    mean = np.array([float(values[name]) for name in names])
    matrix = covariance.loc[names, names].to_numpy(dtype=float)
    matrix = (matrix + matrix.T) / 2          # symetrise les erreurs d'arrondi
    generator = np.random.default_rng(seed)
    sample = generator.multivariate_normal(mean, matrix, size=count, method='eigh')
    return [dict(zip(names, draw)) for draw in sample]


def marginal_effects(the_biogeme, results, variables=None, outcome_labels=None,
                     choice_column='severity', draws=KRINSKY_ROBB_DRAWS,
                     level=CONFIDENCE_LEVEL, seed=0):
    """Effets marginaux moyens et intervalle de confiance, en points de %.

    Une ligne par variable, pour chaque modalite l'effet et son intervalle de
    Krinsky-Robb. `variables` limite le tableau aux variables voulues ; None
    prend toutes celles du modele.
    """
    data = the_biogeme.database.data.copy()
    betas = results.get_beta_values()
    beta_draws = _beta_draws(results, draws, seed) if draws else []
    outcomes = sorted(int(value) for value in data[choice_column].unique())
    labels = outcome_labels or {outcome: f'P(severity = {outcome})'
                                for outcome in outcomes}

    available = _model_variables(the_biogeme, exclude=(choice_column,))

    def normalise(entry):
        """'colonne' ou (libelle, [colonnes]) -> (libelle, colonnes retenues)."""
        if isinstance(entry, str):
            return entry, [entry]
        label, columns = entry
        return label, list(columns)

    if variables is None:
        selection = [(name, [name]) for name in available]
    else:
        selection = []
        for entry in variables:
            label, columns = normalise(entry)
            kept = [column for column in columns if column in available]
            missing = [column for column in columns if column not in available]
            if missing:
                print(f'  [{label}] hors du modele, ignoree : {", ".join(missing)}')
            if kept:
                selection.append((label, kept))

    tail = (1 - level) / 2
    records = []
    for label, columns in selection:
        column = data[columns[0]]
        values = set(pd.concat([data[name] for name in columns]).dropna().unique())
        if values <= {0, 1}:
            # Groupe d'indicatrices : « aucune » contre « la premiere ». Mettre
            # toutes les colonnes a 1 donnerait 2 sur un terme somme, et n'a pas
            # de sens pour des modalites qui s'excluent (nuit avec / sans
            # eclairage).
            scale, kind = 1.0, 'dummy (0 to 1)'
            frame_low = data.assign(**{name: 0.0 for name in columns})
            frame_high = data.assign(**{name: 1.0 if name == columns[0] else 0.0
                                        for name in columns})
        elif column.nunique() <= DISCRETE_MAX_LEVELS:
            # Comptage : une unite de plus, pas une derivee locale extrapolee.
            scale, kind = 1.0, 'per unit (+1)'
            frame_low = data
            frame_high = data.assign(**{columns[0]: column + 1})
        else:
            if len(columns) > 1:
                print(f'  [{label}] groupe continu non gere, ignoree')
                continue
            step = DERIVATIVE_STEP * float(column.std())
            if not step:
                continue                       # variable constante : pas d'effet
            scale, kind = 2 * step, 'per unit'
            frame_low = data.assign(**{columns[0]: column - step})
            frame_high = data.assign(**{columns[0]: column + step})

        low = _outcome_simulators(the_biogeme, frame_low, outcomes, choice_column)
        high = _outcome_simulators(the_biogeme, frame_high, outcomes, choice_column)

        def effect(parameters, outcome):
            difference = (high[outcome].simulate(parameters)['probability']
                          - low[outcome].simulate(parameters)['probability'])
            return 100 * float(difference.mean()) / scale

        record = {'Variable': _pretty_variable(label), 'Variation': kind}
        for outcome in outcomes:
            record[labels[outcome]] = effect(betas, outcome)
            if beta_draws:
                simulated = np.array([effect(draw, outcome) for draw in beta_draws])
                record[f'{labels[outcome]} CI'] = (
                    float(np.quantile(simulated, tail)),
                    float(np.quantile(simulated, 1 - tail)))
        records.append(record)

    table = pd.DataFrame.from_records(records).set_index('Variable')
    # Les variables les plus « actives » d'abord.
    point_columns = [labels[outcome] for outcome in outcomes]
    ordering = table[point_columns].abs().max(axis=1)
    table = table.loc[ordering.sort_values(ascending=False).index]

    # Probabilites moyennes predites sur les donnees non perturbees.
    simulators = _outcome_simulators(the_biogeme, data, outcomes, choice_column)
    baseline = {'Variable': BASELINE_LABEL, 'Variation': 'baseline'}
    for outcome in outcomes:
        baseline[labels[outcome]] = 100 * float(
            simulators[outcome].simulate(betas)['probability'].mean())
    baseline = pd.DataFrame.from_records([baseline]).set_index('Variable')
    return pd.concat([baseline, table])


SEVERITY_LABELS = {1: 'No injury', 2: 'Injury', 3: 'Fatality'}

marginal_effects_tables = {}
for label, (results, the_biogeme, segment) in MAIN_MODELS.items():
    print(f'=== {label} : average marginal effects (percentage points) ===')
    try:
        table = marginal_effects(the_biogeme, results,
                                 variables=KEY_VARIABLES.get(label),
                                 outcome_labels=SEVERITY_LABELS)
    except Exception as error:
        print(f'  calcul impossible : {type(error).__name__}: {error}\n')
        continue
    marginal_effects_tables[label] = table
    display_columns = [column for column in table.columns
                       if not column.endswith(' CI')]
    print(table[display_columns].round(2).to_string())
    print()


# --- Same tables, in LaTeX ----------------------------------------------------
# `_escape`, `_row` and `_pretty` come from the LaTeX cell above.
def make_marginal_effects_table(table, caption=None, label=None,
                                level=CONFIDENCE_LEVEL):
    """Tableau LaTeX des effets marginaux moyens (points de pourcentage)."""
    outcomes = [column for column in table.columns
                if column != 'Variation' and not column.endswith(' CI')]
    columns = ['Variable'] + outcomes
    alignment = 'l' + ' c' * len(outcomes)

    lines = [
        r'\begin{table}[H]', r'\centering', r'\small',
        rf'\caption{{{caption}}}' if caption else r'\caption{}',
        rf'\label{{{label}}}' if label else '',
        rf'\begin{{tabular}}{{{alignment}}}',
        r'\hline',
        _row([_escape(name) for name in columns]),
        r'\hline',
    ]
    for variable, record in table.iterrows():
        is_baseline = record.get('Variation') == 'baseline'
        cells = [r'\quad \textit{' + _escape(str(variable)) + '}' if is_baseline
                 else _escape(str(variable))]
        for outcome in outcomes:
            value = f'{float(record[outcome]):.2f}'
            interval = record.get(f'{outcome} CI')
            if isinstance(interval, tuple) and not is_baseline:
                value += f' [{interval[0]:.2f}; {interval[1]:.2f}]'
            cells.append(value)
        lines.append(_row(cells))
        if is_baseline:
            lines.append(r'\hline')
    interval_note = (', with ' + f'{level:.0%}'.replace('%', r'\%')
                     + r' Krinsky--Robb confidence intervals in brackets'
                     if any(column.endswith(' CI') for column in table.columns)
                     else '')
    lines += [r'\hline', r'\end{tabular}', r'\medskip',
              r'\footnotesize Average marginal effects, in percentage points'
              + interval_note +
              r'. For a dummy, the effect of '
              r'moving it from 0 to 1; for a count, the effect of one more unit '
              r'($x \\rightarrow x+1$); for a continuous variable, the '
              r'derivative. Averaged over the estimation sample. The first '
              r'row gives the mean predicted probability of each outcome, '
              r'against which the effects are to be read.',
              r'\end{table}']
    return '\n'.join(line for line in lines if line)


marginal_effects_tex = {}
for label, table in marginal_effects_tables.items():
    stem = re.sub(r'[^a-z0-9]+', '_', label.lower()).strip('_')
    tex = make_marginal_effects_table(
        table,
        caption=(f'{display_name(label)}: average marginal effects '
                 f'(percentage points).'),
        label=f'tab:marginal_effects_{stem}',
    )
    (AIC_TABLES_DIRECTORY / f'marginal_effects_{stem}.tex').write_text(colorize(tex),
                                                                      encoding='utf-8')
    marginal_effects_tex[label] = tex
    print(AIC_TABLES_DIRECTORY / f'marginal_effects_{stem}.tex')


# --- The four models in a single table -----------------------------------------
# One block per segment, the same three columns throughout: more compact than
# four separate tables, and it makes the segments directly comparable. The
# pedestrian and MMV models merge injury and fatality, so their effect spans
# both columns instead of being printed twice.
def make_combined_marginal_effects_table(
    tables, outcomes=None,
    caption=('Average marginal effects (percentage points) of the key variables, '
             'by segment.'),
    label='tab:marginal_effects_all',
    level=CONFIDENCE_LEVEL,
):
    outcomes = outcomes or list(SEVERITY_LABELS.values())
    columns = ['Variable'] + outcomes
    width = len(columns)

    lines = [
        r'\begin{table}[H]', r'\centering', r'\small',
        r'\setlength{\tabcolsep}{4pt}',
        rf'\caption{{{caption}}}', rf'\label{{{label}}}',
        rf'\begin{{tabular}}{{l{" c" * len(outcomes)}}}',
        r'\hline',
        _row([_escape(name) for name in columns]),
        r'\hline',
    ]

    def _formatted(record, outcome, is_baseline):
        if outcome not in record:
            return None
        value = f'{float(record[outcome]):.2f}'
        interval = record.get(f'{outcome} CI')
        if isinstance(interval, tuple) and not is_baseline:
            value += f' [{interval[0]:.2f}; {interval[1]:.2f}]'
        return value

    def _merges_harmful_outcomes(table):
        """Le segment a-t-il ete estime sur la reponse fusionnee ?

        Les modeles pieton et engin-engin ne distinguent pas la blessure du
        deces : trop peu de deces pour identifier un parametre propre. Leur
        effet marginal est donc le MEME nombre dans les deux colonnes, et le
        repeter donne a lire deux resultats la ou il n'y en a qu'un. On ne se
        fie pas au nom du segment mais a ce que contient la table : les deux
        colonnes doivent coincider sur TOUTES les lignes, faute de quoi une
        egalite fortuite a deux decimales ferait fusionner un vrai modele a
        trois issues.
        """
        if len(outcomes) < 3:
            return False
        injury, fatality = outcomes[1], outcomes[2]
        if injury not in table.columns or fatality not in table.columns:
            return False
        return bool((table[injury] == table[fatality]).all())

    for position, (segment, table) in enumerate(tables.items()):
        if position:
            lines.append(r'\hline')
        lines.append(rf'\multicolumn{{{width}}}{{l}}'
                     rf'{{\textit{{{_escape(display_name(segment))}}}}} \\')
        merged = _merges_harmful_outcomes(table)
        for variable, record in table.iterrows():
            is_baseline = record.get('Variation') == 'baseline'
            cells = [r'\quad \textit{' + _escape(str(variable)) + '}' if is_baseline
                     else _escape(str(variable))]
            for outcome in outcomes:
                value = _formatted(record, outcome, is_baseline)
                if merged and outcome == outcomes[1]:
                    # Une seule valeur, centree sur les deux colonnes.
                    cells.append(r'\multicolumn{2}{c}{' + (value or '') + '}')
                    continue
                if merged and outcome == outcomes[2]:
                    continue
                cells.append('' if value is None else value)
            lines.append(_row(cells))

    has_intervals = any(column.endswith(' CI')
                        for table in tables.values() for column in table.columns)
    interval_note = (', with ' + f'{level:.0%}'.replace('%', r'\%')
                     + r' Krinsky--Robb confidence intervals in brackets'
                     if has_intervals else '')
    lines += [r'\hline', r'\end{tabular}', r'\medskip',
              r'\footnotesize Average marginal effects, in percentage points'
              + interval_note +
              r'. For a dummy, the effect of '
              r'moving it from 0 to 1; for a count, the effect of one more unit '
              r'($x \\rightarrow x+1$); for a continuous variable, the '
              r'derivative. Averaged over each estimation sample. The '
              r'first row of each block gives the mean predicted probability '
              r'of each outcome. Where a single figure spans the injury and '
              r'fatality columns, the model was estimated on the merged '
              r'outcome: those segments hold too few fatalities to identify a '
              r'separate parameter, so one effect applies to both.',
              r'\end{table}']
    return '\n'.join(lines)


combined_marginal_effects_tex = make_combined_marginal_effects_table(
    marginal_effects_tables)
combined_path = AIC_TABLES_DIRECTORY / 'marginal_effects_all.tex'
combined_path.write_text(colorize(combined_marginal_effects_tex),
                         encoding='utf-8')
print()
print(combined_path)
print(combined_marginal_effects_tex)

=== Car crashes : average marginal effects (percentage points) ===
                                       Variation  No injury  Injury  Fatality
Variable                                                                     
Baseline predicted probabilities        baseline       3.59   95.71      0.69
Passenger (vs. driver)            dummy (0 to 1)      13.64  -16.28      2.63
Second-party light vehicle        dummy (0 to 1)      10.30  -12.49      2.18
Second-party heavy vehicle        dummy (0 to 1)      -0.58   -6.86      7.43
Number of vehicles involved        per unit (+1)       4.30   -5.13      0.82
Gender: female                    dummy (0 to 1)      -1.78    2.13     -0.34
Second-party turning right        dummy (0 to 1)      -0.08   -1.14      1.23
Away from an intersection         dummy (0 to 1)      -1.00    0.98      0.02
Crash at night                    dummy (0 to 1)      -0.05   -0.66      0.71
Presence of a cycle facility      dummy (0 to 1)       0.03    0.44     -0.

## Framework comparison

All candidate models side by side: sample size, number of parameters,
log-likelihood, AIC and BIC, with the $\Delta$ of each criterion measured
against the best model **within each sample**. Non-nested frameworks (MNL vs.
ordered probit vs. ordered logit) cannot be compared with a likelihood-ratio
test, which is why the arbitration relies on the information criteria; the
nested comparisons are handled in the LR-test section below.


In [36]:
# --- Framework comparison: N / K / LL / AIC / BIC of every candidate model ----
# Assembles the fit statistics of the models listed in `ESTIMATED_MODELS` into a
# single table, one block per sample, and renders it with
# `latex_tables.make_model_comparison_table` (the deltas and the bold best value
# of each criterion are computed there).

import math
import re
from pathlib import Path

from latex_tables import colorize, make_model_comparison_table

SAMPLE_ORDER = ['Car crashes', 'MMV', 'Pedestrian', 'Single-vehicle']

COMPARISON_TEX = RESULTS_ROOT / 'model_framework_comparison.tex'
COMPARISON_CSV = RESULTS_ROOT / 'model_framework_comparison.csv'

# Les candidats : (echantillon, framework, resultats, fichier).
# Note MMV : avec deux modalites, l'ordered logit EST le logit binaire deja
# estime (meme vraisemblance, memes criteres) ; il n'apparait donc pas ici, ou
# il ne ferait que dupliquer une ligne du tableau.
ESTIMATED_MODELS = [
    ('Car crashes', 'MNL', results_ml_carcrashes),
    ('Car crashes', 'Ordered probit', results_car_probit),
   # ('Car crashes', 'Ordered logit', results_car_logit),
    ('MMV', 'Binary logit', results_logit_mmv),
    ('MMV', 'Binary probit', results_mmv_probit),
  #  ('Pedestrian', 'Ordered logit', results_pedes),
    ('Pedestrian', 'Ordered probit', results_pedes_probit),
    ('Pedestrian', 'Binary logit', results_pedestrian_mnl),
    ('Single-vehicle', 'Ordered probit', results_solo_2),
   # ('Single-vehicle', 'Ordered logit', results_solo_logit),
    ('Single-vehicle', 'MNL', results_solo_mnl),
]


# Constantes alternatives-specifiques et seuils des modeles ordonnes. Le `K`
# AFFICHE les exclut : ce sont les parametres du modele de reference, et ce qui
# distingue deux specifications ce sont les coefficients ajoutes par-dessus.
#
# ATTENTION : l'AIC et le BIC des colonnes voisines utilisent, eux, le K TOTAL,
# parce que c'est leur definition. Un lecteur qui recalculerait 2K - 2LL a partir
# de la colonne K ne retrouverait donc pas l'AIC affiche -- d'ou la note de bas
# de tableau.
CONSTANT_PARAMETER = re.compile(r'^(asc|constant|tau)', re.IGNORECASE)


def _count_constants(names):
    """Nombre de constantes et de seuils parmi des noms de parametres."""
    return sum(1 for name in names if CONSTANT_PARAMETER.match(str(name)))


def _fit_statistics(results):
    """Statistiques d'ajustement d'un objet Biogeme, quel que soit le nommage."""
    data = results.data

    def pick(*names):
        for name in names:
            value = getattr(data, name, None)
            if value is not None:
                return value
        return None

    try:
        constants = _count_constants(results.get_estimated_parameters().index)
    except Exception:                       # noqa: BLE001
        constants = 0

    return {'N': pick('sampleSize', 'sample_size'),
            'K': pick('nparam', 'number_of_parameters'),
            'K constants': constants,
            'LL': pick('logLike', 'final_log_likelihood'),
            'AIC': pick('akaike', 'akaike_information_criterion'),
            'BIC': pick('bayesian', 'bayesian_information_criterion')}


# --- Mixed logits estimated in a separate session -----------------------------
# They live as Biogeme HTML reports in `mixed_model/` rather than as result
# objects, so their statistics are read back from the files.
#
# One caveat drives the code below: under `panel()` Biogeme reports the number of
# PANEL GROUPS as the sample size -- accidents, not rows. Its BIC is therefore
# computed on ln(accidents) while every other row of the table uses ln(rows), and
# the two are not comparable. The BIC is recomputed here on the row count of the
# corresponding estimation sample. The AIC needs no such correction: 2K - 2LL
# does not involve N.

MIXED_LOGIT_DIRECTORY = Path('mixed_model')

# (echantillon, motif du fichier, cadre d'estimation).
MIXED_LOGIT_MODELS = [
    ('Car crashes', 'mixed_logit_car_crashes_panel', df_carcrashes),
    ('MMV', 'mixed_logit_mmv_panel', df_mmv),
    ('Pedestrian', 'mixed_logit_pedestrian_panel', df_pedestrian),
    ('Single-vehicle', 'mixed_logit_sinv_panel', df_sv),
]


def _read_biogeme_html(path):
    """Statistiques d'ajustement lues dans un rapport HTML de Biogeme."""
    text = path.read_text(encoding='utf-8', errors='replace')

    def field(label):
        match = re.search(re.escape(label) + r'.{0,120}?<td[^>]*>(.*?)</td>',
                          text, re.S)
        if match is None:
            raise ValueError(f'{label!r} introuvable dans {path.name}')
        return float(re.sub(r'<[^>]+>', '', match.group(1)).replace(',', '').strip())

    # Uniquement le tableau des parametres estimes : le rapport contient aussi
    # une matrice de correlations dont chaque ligne commence par un nom de
    # parametre, et la compter reviendrait a additionner les memes noms 9 fois.
    start = text.find('<h1>Estimated parameters</h1>')
    block = text[start:text.find('</table>', start)] if start >= 0 else ''
    names = re.findall(r'<tr class=biostyle><td>([^<]+)</td>', block)

    return {'K': int(field('Number of estimated parameters')),
            'K constants': _count_constants(names),
            'groups': int(field('Sample size')),
            'LL': field('Final log likelihood'),
            'AIC': field('Akaike Information Criterion'),
            'draws': int(field('Number of draws'))}


def mixed_logit_rows(models=None, directory=MIXED_LOGIT_DIRECTORY):
    """Une ligne de comparaison par mixed logit trouve dans `directory`."""
    models = MIXED_LOGIT_MODELS if models is None else models
    records = []
    for sample, pattern, frame in models:
        matches = sorted(directory.glob(f'{pattern}*.html'))
        if not matches:
            print(f'  [{sample}] aucun fichier {pattern}*.html, ignore')
            continue
        path = matches[-1]          # le suffixe ~NN le plus eleve est le dernier run
        statistics = _read_biogeme_html(path)

        rows = len(frame)
        # Verification d'integrite : le nombre de groupes du fichier doit etre le
        # nombre d'accidents du cadre. S'il differe, le modele n'a pas ete estime
        # sur cet echantillon et son BIC recalculte serait faux.
        crashes = frame['Num_Acc'].nunique()
        if statistics['groups'] != crashes:
            print(f'  [{sample}] ATTENTION {path.name} : {statistics["groups"]} groupes '
                  f'contre {crashes} accidents dans le cadre courant -- '
                  f'echantillon different, BIC non recalcule')
            rows = None

        parameters, loglikelihood = statistics['K'], statistics['LL']
        bic = (math.log(rows) * parameters - 2 * loglikelihood
               if rows else float('nan'))
        records.append({
            'Sample': sample, 'Framework': 'Mixed logit (crash panel)',
            'N': rows if rows else statistics['groups'],
            'K': parameters - statistics['K constants'],
            'LL': loglikelihood,
            'AIC': statistics['AIC'], 'BIC': bic,
        })
        print(f'  [{sample}] {path.name} : K={parameters} LL={loglikelihood:.3f} '
              f'draws={statistics["draws"]}')
    return records


def build_comparison_table(models=None, sample_order=None,
                           include_mixed_logit=True):
    """DataFrame Sample / Framework / N / K / LL / AIC / BIC + les deltas.

    Les deltas sont mesures a l'interieur de chaque echantillon, contre le
    meilleur modele au sens du critere considere. Les lignes sont triees par
    echantillon (`sample_order`) puis par BIC croissant.
    """
    models = ESTIMATED_MODELS if models is None else models
    sample_order = SAMPLE_ORDER if sample_order is None else sample_order

    records = []
    for sample, framework, results in models:
        fit = _fit_statistics(results)
        records.append({'Sample': sample, 'Framework': framework,
                        'N': int(fit['N']),
                        'K': int(fit['K']) - int(fit['K constants']),
                        'LL': float(fit['LL']), 'AIC': float(fit['AIC']),
                        'BIC': float(fit['BIC'])})
    if include_mixed_logit:
        records.extend(mixed_logit_rows())
    table = pd.DataFrame.from_records(records)

    for criterion in ('AIC', 'BIC'):
        best = table.groupby('Sample')[criterion].transform('min')
        table[f'Delta {criterion}'] = table[criterion] - best

    # Ordre des echantillons impose ; un echantillon absent de la liste passe
    # a la fin plutot que de disparaitre.
    rank = {sample: position for position, sample in enumerate(sample_order)}
    table['_rank'] = table['Sample'].map(lambda s: rank.get(s, len(rank)))
    table = (table.sort_values(['_rank', 'BIC'])
                  .drop(columns='_rank')
                  .reset_index(drop=True))
    return table


comparison_table = build_comparison_table()
comparison_table.to_csv(COMPARISON_CSV, index=False)

comparison_tex = make_model_comparison_table(comparison_table,
                                             sample_order=SAMPLE_ORDER)
COMPARISON_TEX.write_text(colorize(comparison_tex), encoding='utf-8')

print(f'{COMPARISON_CSV}\n{COMPARISON_TEX}\n')
display(comparison_table.round(2))
print(comparison_tex)

  [Car crashes] mixed_logit_car_crashes_panel~back.html : K=17 LL=-1447.006 draws=5000
  [MMV] mixed_logit_mmv_panel~02.html : K=11 LL=-688.760 draws=5000
  [Pedestrian] mixed_logit_pedestrian_panel~03.html : K=10 LL=-1057.999 draws=10000
  [Single-vehicle] mixed_logit_sinv_panel~01.html : K=8 LL=-284.858 draws=5000
results/model_framework_comparison.csv
results/model_framework_comparison.tex



,Sample,Framework,N,K,LL,AIC,BIC,Delta AIC,Delta BIC
0,Car crashes,MNL,9101,15,-1445.63,2925.26,3046.24,0.00,0.00
1,Car crashes,Mixed logit (crash panel),9101,15,-1447.01,2928.01,3048.99,2.75,2.75
2,Car crashes,Ordered probit,9101,16,-1489.70,3015.40,3143.49,90.14,97.26
3,MMV,Binary probit,1282,11,-665.45,1354.90,1416.77,0.00,0.00
4,MMV,Binary logit,1282,10,-686.20,1394.41,1451.12,39.51,34.35
5,MMV,Mixed logit (crash panel),1282,10,-688.76,1399.52,1456.24,44.62,39.47
6,Pedestrian,Binary logit,2792,8,-1058.00,2134.00,2187.41,0.00,0.00
7,Pedestrian,Mixed logit (crash panel),2792,9,-1058.00,2136.00,2195.34,2.00,7.94
8,Pedestrian,Ordered probit,2792,8,-1124.25,2268.51,2327.85,134.51,140.44
9,Single-vehicle,MNL,2212,5,-284.86,583.72,623.63,0.00,0.00


\begin{table}[htbp]
\centering
\small
\caption{Goodness-of-fit of the estimated frameworks, by sample. $\Delta$ is measured against the best model within each sample; the best value of each criterion is in bold.}
\label{tab:model_comparison}
\begin{tabular}{l r r r r r r r}
\hline
Framework & N & K & LL & AIC & $\Delta$AIC & BIC & $\Delta$BIC \\
\hline
\multicolumn{8}{l}{\textit{Car crashes}} \\
MNL & 9101 & 15 & -1445.63 & \textbf{2925.26} & 0.00 & \textbf{3046.24} & 0.00 \\
Mixed logit (crash panel) & 9101 & 15 & -1447.01 & 2928.01 & 2.75 & 3048.99 & 2.75 \\
Ordered probit & 9101 & 16 & -1489.70 & 3015.40 & 90.14 & 3143.49 & 97.26 \\
\hline
\multicolumn{8}{l}{\textit{MMV}} \\
Binary probit & 1282 & 11 & -665.45 & \textbf{1354.90} & 0.00 & \textbf{1416.77} & 0.00 \\
Binary logit & 1282 & 10 & -686.20 & 1394.41 & 39.51 & 1451.12 & 34.35 \\
Mixed logit (crash panel) & 1282 & 10 & -688.76 & 1399.52 & 44.62 & 1456.24 & 39.47 \\
\hline
\multicolumn{8}{l}{\textit{Pedestrian}} \\
Binary logi

## Poolability: interactions by crash type

Two models are estimated on the **same** pooled data, both carrying
segment-specific constants so that the test bears on the slopes alone:

- **pooled**: each variable of the common specification enters with a single
  coefficient, shared by every segment;
- **interacted**: the same, plus a *deviation* for every variable in every
  segment but one, taken as reference.

$$\chi^2 = 2\left[\ell_{\text{interacted}} - \ell_{\text{pooled}}\right],
\qquad \text{d.f.} = \text{number of deviations}$$

Because every variable is interacted with every segment, the unrestricted model
is **equivalent to estimating the common specification separately on each
segment**: the effect it implies for a segment reproduces that segment's own
estimate, and each deviation reads as "this segment departs from the reference
coefficient". This is the poolability test in its strict form (Chow; Ben-Akiva
and Lerman, 1985).

Two constructions are deliberate. The reference segment carries no deviation --
the common effect is already its effect, and adding one would make the common
column collinear with the sum of the interactions, which inflates the joint Wald
statistic through the inversion of a singular matrix. And a deviation is created
only where the variable actually varies against the outcome inside the segment,
which is what keeps the small single-vehicle segment estimable.

The test runs on the injured / not-injured margin. On the fatality margin,
several variables have no death at all in a segment, which separates them
perfectly and makes the coefficients -- and the Wald statistic -- meaningless.


In [37]:
# --- Poolability: interactions by crash type, on the real specifications -------
# Two models estimated on the SAME pooled data, both with a segment-specific
# constant, so that the test isolates the SLOPES:
#
#   - pooled     : each variable enters with a SINGLE coefficient, common to
#     every segment;
#   - interacted : the same, PLUS a deviation for EVERY variable in EVERY segment
#     but one, taken as reference.
#
# Giving every variable a deviation in every segment makes the unrestricted model
# strictly equivalent to estimating that common specification separately on each
# segment: the effect implied for a segment reproduces its separate estimate, and
# a deviation reads as "this segment departs from the reference coefficient".
# That is the poolability test in its strict form (Chow; Ben-Akiva and Lerman
# 1985).
#
# The deviation form matters. Letting a variable enter ONLY in its own segment
# would not nest the pooled model -- the pooled model gives that variable an
# effect in the other segments too -- and the likelihood-ratio statistic would
# come out negative, as it did in a first attempt. Written as common effect plus
# deviation, the restriction "all deviations are zero" is exactly the pooled
# model, so the test is valid and the statistic positive.
#
#     chi2 = 2 [ LL_interacted - LL_pooled ] ,   df = number of deviations
#
# The reference segment carries NO deviation: the common effect is already its
# effect. Creating one would make the common column collinear with the sum of the
# interactions -- a redundant parameter, a singular covariance, and a joint Wald
# statistic inflated by the inversion of a near-singular matrix.
#
# The test runs on the injured / not-injured margin. The fatality margin was
# tried and dropped: with about 97 deaths spread over three segments, several
# variables have no fatality at all in a segment -- perfect separation, huge
# coefficients, and a Wald statistic that explodes while the LR sees nothing.

import numpy as np
from scipy.stats import chi2 as chi2_distribution, norm as normal_distribution

# Les variables retenues dans chaque specification estimee. A ajuster si tu
# modifies un modele : ce sont des noms de COLONNES, pas des noms de Beta.
SEGMENT_SPECIFICATIONS = {
    'Car crashes': [
        'age', 'Gender_Female', 'User category_Passenger',
        'number of involved vehicles', 'Point of impact_Back',
        'Intersection_No intersection', 'Maneuver_2_Turning right',
        'Lighting conditions_Night with street lightings on',
        'Accident location_On cycle facility',
    ],
    'MMV': [
        'age', 'Gender_Female', 'Surface condition_Wet', 'Maneuver_Swerving',
        'Maneuver_Turning left', 'Point of impact_Back', 'Gender_2_Female',
        'age_opposite_mean',
    ],
    'Pedestrian': [
        'age', 'Gender_Female', 'User category_Pedestrian',
        'Crossroad_Traffic lights', 'Intersection_No intersection',
        'Gender_2_Female', 'age_opposite_mean',
    ],
    'Single-vehicle': [
        'age', 'User category_Passenger', 'Long profile_Slope',
        'Number of passengers',
    ],
}

POOLABILITY_FRAMES = {
    'Car crashes': df_carcrashes, 'MMV': df_mmv,
    'Pedestrian': df_pedestrian, 'Single-vehicle': df_sv,
}

# (libelle, modalite consideree comme l'evenement)
POOLABILITY_OUTCOMES = {
    'Injury (severity > 1)': lambda severity: severity > 1,
}
OUTCOME_COLUMN = 'outcome'          # 1 = non, 2 = oui


def _pooled_frame(frames, event):
    """Donnees empilees, avec une indicatrice par segment et la variable expliquee."""
    pieces = []
    for name, frame in frames.items():
        piece = frame.select_dtypes(include='number').copy()
        piece[OUTCOME_COLUMN] = event(frame['severity']).astype(int) + 1
        for other in frames:
            piece[f'segment_{_pool_slug(other)}'] = int(other == name)
        pieces.append(piece)
    return pd.concat(pieces, ignore_index=True)


def _pool_slug(text):
    return re.sub(r'[^a-z0-9]+', '_', str(text).lower()).strip('_')


def _estimate(data, utility, suffix):
    logprob = models.loglogit({1: 0, 2: utility}, {1: 1, 2: 1},
                              Variable(OUTCOME_COLUMN))
    the_biogeme = bio.BIOGEME(db.Database(f'pool_{suffix}', data), logprob,
                              generate_html=False, generate_pickle=False)
    the_biogeme.modelName = f'pool_{suffix}'
    return the_biogeme.estimate()


def _usable(data, column, segment_column=None):
    """La colonne varie-t-elle la ou elle est utilisee, et l'evenement aussi ?"""
    mask = data[segment_column] > 0 if segment_column else pd.Series(True, index=data.index)
    piece = data.loc[mask]
    if piece[column].nunique() < 2 or piece[OUTCOME_COLUMN].nunique() < 2:
        return False
    # Le controle de separation ne vaut que pour les indicatrices : sur une
    # variable continue, `colonne > 0` est constante et rejetterait tout.
    if set(piece[column].dropna().unique()) <= {0, 1}:
        table = pd.crosstab(piece[column] > 0, piece[OUTCOME_COLUMN])
        return table.size == 4 and table.to_numpy().min() >= 1
    return True


def poolability_by_interaction(frames=None, specifications=None, event=None,
                               suffix='injury', verbose=True):
    """Modele interagi par segment contre modele a coefficients communs."""
    frames = POOLABILITY_FRAMES if frames is None else frames
    specifications = SEGMENT_SPECIFICATIONS if specifications is None else specifications
    data = _pooled_frame(frames, event)

    # Constantes propres aux segments dans LES DEUX modeles : le test ne porte
    # que sur les pentes.
    constants = 0
    for name in list(frames)[1:]:                     # le premier segment est la reference
        constants = constants + (Beta(f'asc_{_pool_slug(name)}_{suffix}', 0, None, None, 0)
                                 * Variable(f'segment_{_pool_slug(name)}'))
    intercept = Beta(f'asc_{suffix}', 0, None, None, 0)

    # L'union des variables retenues par au moins une specification. C'est elle
    # qui forme le modele poole : une specification unique, appliquee a tous.
    dropped, union = {}, []
    for name, columns in specifications.items():
        if name not in frames:
            continue
        for column in columns:
            if column not in data.columns:
                dropped[f'{name} / {column}'] = 'colonne absente'
                continue
            if column not in union:
                union.append(column)

    # Un ecart par variable ET par segment, et non plus seulement dans le segment
    # dont la specification l'avait retenue. Le modele non contraint est alors
    # exactement equivalent a quatre logits estimes separement sur cette
    # specification commune : l'effet implicite dans un segment reproduit son
    # estimation separee, et l'ecart se lit vraiment comme « ce segment s'ecarte
    # du coefficient de reference ». C'est le test de poolabilite au sens strict
    # (Chow ; Ben-Akiva et Lerman 1985).
    #
    # Le premier segment ou la variable est estimable sert de REFERENCE : son
    # ecart n'est pas cree, car l'effet commun est deja son effet. Sans cela les
    # colonnes seraient colineaires -- la colonne commune est la somme des
    # colonnes d'interaction -- et le Wald joint inverserait une matrice
    # singuliere. Une variable estimable dans un seul segment n'a donc aucun
    # ecart : il n'y a rien a comparer.
    kept, reference, deviations = {}, {}, []
    for column in union:
        active = [name for name in frames
                  if _usable(data, column, f'segment_{_pool_slug(name)}')]
        for name in frames:
            if name not in active:
                dropped[f'{name} / {column}'] = (
                    'aucune variation utilisable dans ce segment')
        if not active:
            continue
        reference[column] = active[0]
        for name in active:
            kept.setdefault(name, []).append(column)
            if name != active[0]:
                deviations.append((name, column, f'segment_{_pool_slug(name)}'))
    kept = {name: kept.get(name, []) for name in frames}

    # Effet commun a tous les segments : c'est le modele restreint.
    common = intercept + constants
    for column in union:
        common = common + (Beta(f'beta_{_pool_slug(column)}_common_{suffix}',
                                0, None, None, 0) * Variable(column))
    pooled = common

    # Modele non contraint : effet commun + ecart propre au segment qui a
    # selectionne la variable.
    interacted = common
    for name, column, segment_column in deviations:
        interacted = interacted + (
            Beta(f'delta_{_pool_slug(column)}_{_pool_slug(name)}_{suffix}',
                 0, None, None, 0)
            * Variable(column) * Variable(segment_column))

    interacted_results = _estimate(data, interacted, f'{suffix}_interacted')
    pooled_results = _estimate(data, pooled, f'{suffix}_pooled')
    interacted_fit = _fit_statistics(interacted_results)
    pooled_fit = _fit_statistics(pooled_results)

    statistic = 2 * (float(interacted_fit['LL']) - float(pooled_fit['LL']))
    degrees = int(interacted_fit['K']) - int(pooled_fit['K'])
    p_value = float(chi2_distribution.sf(statistic, degrees))

    # Wald joint sur les memes ecarts, a partir de la covariance robuste du
    # modele non contraint : meme hypothese nulle que le LR, autre statistique.
    # Le t de chaque ecart dans le tableau est le Wald a un degre de liberte.
    values = interacted_results.get_beta_values()
    covariance = interacted_results.get_robust_var_covar()
    names = [f'delta_{_pool_slug(column)}_{_pool_slug(name)}_{suffix}'
             for name, column, _ in deviations]
    names = [name for name in names if name in covariance.index]
    vector = np.array([float(values[name]) for name in names])
    matrix = covariance.loc[names, names].to_numpy(dtype=float)
    # `solve` ne leve pas d'erreur sur une matrice seulement PRESQUE singuliere :
    # elle rend une statistique enorme et fausse. On verifie donc le rang.
    if np.linalg.matrix_rank(matrix) < len(names):
        print(f'  ATTENTION : covariance des ecarts singuliere '
              f'(rang {np.linalg.matrix_rank(matrix)} < {len(names)}), '
              f'Wald joint non calculable.')
        wald, wald_p = float('nan'), float('nan')
    else:
        wald = float(vector @ np.linalg.solve(matrix, vector))
        wald_p = float(chi2_distribution.sf(wald, len(names)))

    if verbose and dropped:
        print('  termes ecartes :')
        for key, reason in dropped.items():
            print(f'    {key:60s} {reason}')

    return {'data': data, 'kept': kept, 'union': union,
            'deviations': deviations, 'reference': reference,
            'wald': wald, 'wald_df': len(names), 'wald_p': wald_p,
            'interacted': interacted_results, 'pooled': pooled_results,
            'interacted_fit': interacted_fit, 'pooled_fit': pooled_fit,
            'chi2': statistic, 'df': degrees, 'p': p_value}


poolability_by_outcome = {}
for outcome_label, event in POOLABILITY_OUTCOMES.items():
    print(f'=== {outcome_label} : {", ".join(POOLABILITY_FRAMES)} ===')
    result = poolability_by_interaction(
        frames=POOLABILITY_FRAMES, event=event, suffix=_pool_slug(outcome_label))
    poolability_by_outcome[outcome_label] = result
    print(f'  interagi : N = {int(result["interacted_fit"]["N"])}, '
          f'K = {int(result["interacted_fit"]["K"])}, '
          f'LL = {float(result["interacted_fit"]["LL"]):.2f}')
    print(f'  poole    : N = {int(result["pooled_fit"]["N"])}, '
          f'K = {int(result["pooled_fit"]["K"])}, '
          f'LL = {float(result["pooled_fit"]["LL"]):.2f}')
    print(f'  LR   : chi2 = {result["chi2"]:8.2f}, df = {result["df"]:3d}, '
          f'p = {result["p"]:.3g}')
    print(f'  Wald : chi2 = {result["wald"]:8.2f}, df = {result["wald_df"]:3d}, '
          f'p = {result["wald_p"]:.3g}')
    print('  ->', 'les pentes different entre segments.'
          if result['p'] < 0.05 else
          'PAS de difference significative des pentes entre segments.')
    print()


# --- The two tests, in LaTeX --------------------------------------------------
SIGNIFICANCE = [(0.01, '***'), (0.05, '**'), (0.10, '*')]


def _stars(t_statistic):
    """Etoiles de significativite du Wald a un degre de liberte : chi2(1) = t^2."""
    p_value = 2 * float(normal_distribution.sf(abs(t_statistic)))
    for threshold, mark in SIGNIFICANCE:
        if p_value < threshold:
            return r'$^{' + mark + r'}$'
    return ''


def _coefficients(results):
    table = results.get_estimated_parameters()
    column = _parameter_columns(table)
    return {name: (float(record[column['value']]), float(record[column['t']]))
            for name, record in table.iterrows()}


def make_poolability_table(result, outcome_label, suffix, label=None):
    """Effet commun et ecarts par segment, une colonne par segment."""
    segments = [name for name in result['kept']]
    coefficients = _coefficients(result['interacted'])
    label = label or f'tab:poolability_{suffix}'

    header = _row([r'\textbf{Variable}', r'\textbf{Common}']
                  + [r'\textbf{' + _escape(name) + '}' for name in segments])
    lines = [
        r'\begin{table}[H]', r'\centering', r'\small',
        r'\setlength{\tabcolsep}{4pt}',
        rf'\caption{{Segment-specific deviations, {outcome_label}. '
        r'Effect in the reference segment and, for every other segment, its '
        r'deviation from that effect (Wald significance of each deviation). '
        r'The unrestricted model is equivalent to estimating the common '
        r'specification separately on each segment.}',
        rf'\label{{{label}}}',
        rf'\begin{{tabular}}{{p{{4cm}} r{" r" * len(segments)}}}',
        r'\hline', header, r'\hline',
    ]

    def cell(estimate, starred=True):
        """Estimation, suivie des etoiles du Wald a un degre de liberte.

        Uniquement dans les colonnes de segment : la l'hypothese nulle est
        `ecart = 0`, soit « ce segment partage le coefficient poole », qui est
        l'objet du tableau. Dans la colonne Common, une etoile testerait
        `coefficient = 0` -- la significativite ordinaire, une tout autre
        hypothese. Deux sens pour un meme symbole induiraient le lecteur en
        erreur, donc la colonne Common reste nue.
        """
        if estimate is None:
            return ''
        return f'{estimate[0]:.3f}' + (_stars(estimate[1]) if starred else '')

    reference = result.get('reference', {})
    for column in result['union']:
        cells = [_escape(column),
                 cell(coefficients.get(f'beta_{_pool_slug(column)}_common_{suffix}'),
                      starred=False)]
        for name in segments:
            if reference.get(column) == name:
                # Segment de reference : son ecart n'est pas identifie, son
                # effet EST l'effet commun.
                cells.append('ref.')
                continue
            key = f'delta_{_pool_slug(column)}_{_pool_slug(name)}_{suffix}'
            cells.append(cell(coefficients.get(key)))
        lines.append(_row(cells))

    interacted, pooled = result['interacted_fit'], result['pooled_fit']
    lines += [
        r'\hline',
        _row(['Observations', f'{int(pooled["N"])}'] + [''] * len(segments)),
        _row(['Parameters (pooled / interacted)',
              f'{int(pooled["K"])} / {int(interacted["K"])}'] + [''] * len(segments)),
        _row(['Log-likelihood (pooled / interacted)',
              f'{float(pooled["LL"]):.2f} / {float(interacted["LL"]):.2f}']
             + [''] * len(segments)),
        r'\hline',
    ]
    note = (r'\footnotesize Likelihood-ratio test of the deviations being jointly '
            rf'zero: $\chi^2 = {result["chi2"]:.1f}$, {result["df"]} d.f., '
            + ('$p < 0.001$' if result['p'] < 0.001 else f'$p = {result["p"]:.3f}$')
            + rf'; joint Wald test on the same deviations: $\chi^2 = '
              rf'{result["wald"]:.1f}$, {result["wald_df"]} d.f., '
            + ('$p < 0.001$' if result['wald_p'] < 0.001
               else f'$p = {result["wald_p"]:.3f}$')
            + r'. Both models carry segment-specific constants, so the test bears '
              r'on the slopes alone. Stars report the Wald test of that single '
              r'deviation being zero, i.e. of that segment sharing the pooled '
              r'coefficient: $^{***}\,p<0.01$, $^{**}\,p<0.05$, $^{*}\,p<0.10$. '
              r'They are reported for the deviations only: in the \emph{Common} '
              r'column a star would test the coefficient being zero, a different '
              r'hypothesis. '
              r'\emph{ref.} marks the reference segment of each '
              r'variable, whose effect is the one reported in the \emph{Common} '
              r'column; a blank cell means the variable could not be estimated in '
              r'that segment, for lack of variation or because every observation '
              r'of one severity level shares the same value.')
    lines += [
        (rf'\multicolumn{{{2 + len(segments)}}}'
         r'{p{\dimexpr\linewidth-2\tabcolsep}}{' + note + r'} \\'),
        r'\end{tabular}', r'\end{table}',
    ]
    return '\n'.join(lines)


poolability_tex = {}
for outcome_label, result in poolability_by_outcome.items():
    suffix = _pool_slug(outcome_label)
    poolability_tex[outcome_label] = tex
    path = AIC_TABLES_DIRECTORY / f'poolability_{suffix}.tex'
    path.write_text(colorize(tex) + '\n', encoding='utf-8')
    print(path)
print()
print(next(iter(poolability_tex.values())))

=== Injury (severity > 1) : Car crashes, MMV, Pedestrian, Single-vehicle ===
  termes ecartes :
    Single-vehicle / Maneuver_2_Turning right                    aucune variation utilisable dans ce segment
    Single-vehicle / Maneuver_Turning left                       aucune variation utilisable dans ce segment
    Single-vehicle / Gender_2_Female                             aucune variation utilisable dans ce segment
    Single-vehicle / age_opposite_mean                           aucune variation utilisable dans ce segment
    Car crashes / User category_Pedestrian                       aucune variation utilisable dans ce segment
    MMV / User category_Pedestrian                               aucune variation utilisable dans ce segment
    Single-vehicle / User category_Pedestrian                    aucune variation utilisable dans ce segment
  interagi : N = 15387, K = 69, LL = -3042.79
  poole    : N = 15387, K = 22, LL = -3170.54
  LR   : chi2 =   255.51, df =  47, p = 1.84e-30


## Manuscript tables

The four estimation tables of the manuscript, written straight from the estimated
models -- same layout, same wording, same significance stars -- so that
re-estimating a model updates the thesis instead of asking for a manual copy.
Each row maps a sentence to the Biogeme parameters that carry it; a parameter
shared by two utilities is printed in both columns and flagged by a footnote,
and any estimated parameter missing from the mapping is appended at the end of
the table rather than silently dropped. Files go to `results/manuscript/`.


In [38]:
# --- Manuscript tables --------------------------------------------------------
# The four estimation tables of the manuscript, written straight from the
# estimated models: same layout, same wording, same significance stars, so that
# re-estimating a model updates the thesis instead of asking for a manual copy.
#
# A row maps a human sentence to the Biogeme parameters that carry it, one per
# column of the table ('Injury' / 'Fatality' for the MNL, a single column for the
# binary and ordered models). `None` prints the dash used when a variable does
# not enter that utility, and the same parameter in two columns is a coefficient
# constrained to be equal -- flagged by a footnote, as in the manuscript.
#
# Any estimated parameter missing from the spec is appended in a final section
# rather than silently dropped: the printed table always accounts for K.

import math

def first_defined(*names):
    """Le premier de ces noms qui existe dans le notebook.

    Les modeles evoluent (l'ordered logit pietonnier a laisse place a l'ordered
    probit) : on nomme les candidats par ordre de preference plutot que de
    referencer une variable qui peut avoir disparu.
    """
    for name in names:
        if name in globals():
            return globals()[name]
    raise NameError(f'aucun de ces resultats n\'est defini : {", ".join(names)}')


MANUSCRIPT_DIRECTORY = RESULTS_ROOT / 'manuscript'
MANUSCRIPT_DIRECTORY.mkdir(parents=True, exist_ok=True)

SIGNIFICANCE = [(0.01, '***'), (0.05, '**'), (0.10, '*')]

THRESHOLD_LABELS = [r"Threshold between `No injury' and `Injury' ($\tau_1$)",
                    r"Threshold between `Injury' and `Fatality' ($\tau_2$)"]


def _number(value, digits=3):
    """3 chiffres significatifs, comme dans le manuscrit (0.0159, -1.32, 6.04)."""
    return f'{float(value):.{digits}g}'


def _stars(p_value):
    if p_value is None or p_value != p_value:
        return ''
    for threshold, mark in SIGNIFICANCE:
        if float(p_value) < threshold:
            return mark
    return ''


def _estimates(results):
    """{nom: (valeur, ecart-type robuste, p-value robuste)}."""
    table = results.get_estimated_parameters()
    column = _parameter_columns(table)
    return {name: (float(record[column['value']]),
                   float(record[column['std']]),
                   float(record[column['p']]))
            for name, record in table.iterrows()}


def _threshold_rows(results, estimates):
    """(libelle, valeur, se, p) pour tau_1 et tau_2 = tau_1 + diff.

    Biogeme estime le premier seuil et l'ecart au suivant ; le second seuil est
    donc une combinaison lineaire, dont l'ecart-type se calcule exactement a
    partir de la covariance robuste.
    """
    names = [name for name in estimates if name.startswith('tau')]
    base = [name for name in names if 'diff' not in name]
    if not base:
        return []

    first = base[0]
    rows = [(THRESHOLD_LABELS[0], *estimates[first])]

    differences = [name for name in names if name.startswith(f'{first}_diff')]
    if differences:
        second = differences[0]
        covariance = results.get_robust_var_covar()
        value = estimates[first][0] + estimates[second][0]
        variance = (float(covariance.loc[first, first])
                    + float(covariance.loc[second, second])
                    + 2 * float(covariance.loc[first, second]))
        std_error = math.sqrt(variance)
        p_value = 2 * (1 - norm.cdf(abs(value / std_error)))
        rows.append((THRESHOLD_LABELS[1], value, std_error, p_value))
    return rows


def make_manuscript_table(results, specification, baseline=None):
    """Le longtable d'estimation et la petite table de statistiques."""
    estimates = _estimates(results)
    columns = specification['columns']
    width = 1 + 2 * len(columns)
    used = set()

    def cells(parameters):
        row = []
        for column in columns:
            name = parameters.get(column)
            if name is None or name not in estimates:
                row += ['-', '-']
                continue
            value, std_error, p_value = estimates[name]
            used.add(name)
            row += [f'{_number(value)} {_stars(p_value)}'.strip(),
                    _number(std_error)]
        return row

    body = []

    for label, value, std_error, p_value in _threshold_rows(results, estimates):
        used.update(name for name in estimates if name.startswith('tau'))
        body.append(_row([label, f'{_number(value)} {_stars(p_value)}'.strip(),
                          _number(std_error)]))

    for section, rows in specification['sections']:
        for row in rows:
            if not (isinstance(row, tuple) and len(row) == 2
                    and isinstance(row[1], dict)):
                raise ValueError(
                    f"section '{section}' : ligne malformee, attendu "
                    f"(libelle, {{colonne: parametre}}), recu {row!r}")
        printed = [(label, parameters) for label, parameters in rows
                   if any(name in estimates for name in parameters.values()
                          if name is not None)]
        if not printed:
            continue
        body += [r'\\', r'{\textbf{' + section + r'}}\\']
        for label, parameters in printed:
            body.append(_row([label] + cells(parameters)))

    # Filet de securite : rien ne disparait du tableau.
    forgotten = [name for name in estimates
                 if name not in used and not name.startswith('tau')]
    if forgotten:
        body += [r'\\', r'{\textbf{Other estimated parameters}}\\']
        for name in forgotten:
            body.append(_row([_escape(name)]
                             + cells({columns[0]: name})))

    if len(columns) == 1:
        alignment = 'p{7cm}p{2cm}p{1.5cm}'
        header = [_row([r'\textbf{Variable}', r'\textit{Estimates}',
                        r'\textit{SE}'])]
    else:
        alignment = 'p{8cm}' + 'p{2cm}p{1.5cm}' * len(columns)
        header = [
            _row([r'\multirow{2}{*}{\textbf{Variable}}']
                 + [r'\multicolumn{2}{c}{\textbf{' + column + '}}'
                    for column in columns]),
            _row([''] + [item for _ in columns
                         for item in (r'\textit{Estimates}', '{SE}')]),
        ]

    significance = (r'\footnotesize Level of significance: * $p<0.10$, '
                    r'** $p<0.05$, *** $p<0.01$.')
    notes = [rf'\multicolumn{{{width}}}{{l}}{{SE: robust standard error}} \\',
             rf'\multicolumn{{{width}}}{{l}}{{' + significance + r'} \\']
    for note in specification.get('notes', []):
        notes.append(rf'\multicolumn{{{width}}}{{p{{\linewidth}}}}{{\footnotesize '
                     + note + r'} \\')

    lines = [
        r'\begin{small}',
        rf'\begin{{longtable}}{{{alignment}}}',
        rf'\caption{{{specification["caption"]}}} \label{{{specification["label"]}}} \\',
        r'\hline', *header, r'\hline', r'\endfirsthead',
        r'{\textit{(continued)}} \\',
        r'\hline', *header, r'\hline', r'\endhead',
        rf'\hline \multicolumn{{{width}}}{{r}}{{\textit{{Continued on next page}}}} \\',
        r'\endfoot',
        r'\hline', r'\endlastfoot',
        *body,
        r'\\', *notes,
        r'\end{longtable}',
        r'\end{small}',
    ]

    fit = _fit_statistics(results)
    baseline_fit = _fit_statistics(baseline) if baseline is not None else None
    null_log_likelihood = (baseline_fit['LL'] if baseline_fit is not None
                           else fit['LL0'])

    # `K` EXCLUT les constantes alternatives-specifiques et les seuils tau.
    # Le modele de reference les depense deja : ce qui distingue les deux
    # modeles, et ce que le rho-barre-deux doit penaliser, ce sont les
    # coefficients ajoutes par-dessus. Le compte est pris sur le modele nul
    # lui-meme plutot que par filtrage des noms, ce qui vaut aussi bien pour un
    # MNL (une constante par alternative) que pour un modele ordonne (les taus).
    #
    # Attention : l'AIC et le BIC du tableau de comparaison des frameworks
    # utilisent, eux, le K TOTAL -- c'est leur definition. Les deux tableaux
    # affichent donc des K differents pour un meme modele, ce qui merite une
    # note dans le manuscrit.
    estimated_constants = int(baseline_fit['K']) if baseline_fit is not None else 0
    reported_parameters = int(fit['K']) - estimated_constants

    statistics = [
        r'\begin{small}', r'\begin{table}[h!]', r'\centering',
        rf'\caption{{{specification["stats_caption"]}}}',
        rf'\label{{{specification["stats_label"]}}}',
        r'\begin{tabular}{l c}', r'\hline',
        _row([r'\textbf{Number of observations}',
              f'{int(fit["N"]):,}'.replace(',', '{,}')]),
        _row([r'\textbf{Number of estimated parameters $K$}',
              f'{reported_parameters}']),
    ]
    if null_log_likelihood is not None:
        statistics.append(_row([r'\textbf{$LL(c)$}',
                                f'{float(null_log_likelihood):,.0f}'.replace(',', '{,}')]))
    statistics.append(_row([r'\textbf{$LL(\hat{\beta})$}',
                            f'{float(fit["LL"]):,.0f}'.replace(',', '{,}')]))
    if null_log_likelihood is not None:
        rho_bar = (1 - (float(fit['LL']) - reported_parameters)
                   / float(null_log_likelihood))
        statistics.append(_row(
            [r'\textbf{$\bar{\rho}^2 = 1 - \dfrac{LL(\hat{\beta}) - K}{LL(c)}$}',
             f'{rho_bar:.3f}']))
    statistics += [r'\hline', r'\end{tabular}', r'\end{table}', r'\end{small}']

    return '\n'.join(lines) + '\n\n' + '\n'.join(statistics)


# --- The four specifications --------------------------------------------------
MANUSCRIPT_SPECIFICATIONS = {
    'Car crashes': {
        'results': lambda: (first_defined('results_ml_carcrashes'),
                            first_defined('results_constant_car')),
        'file': 'model_car.tex',
        'caption': ('Estimation results of the logistic regression model '
                    'predicting the injury severity of MMV riders in crashes '
                    'with motorized vehicles'),
        'label': 'tab:regression_results_car',
        'stats_caption': ('Model statistics of the logistic regression '
                          '(MMV vs. motorized vehicles)'),
        'stats_label': 'tab:regression_results_car_stats',
        'columns': ['Injury', 'Fatality'],
        'sections': [
            ('Alternative specific constants', [
                (r'Alternative specific constant $\alpha$',
                 {'Injury': 'constant_I', 'Fatality': 'constant_F'}),
            ]),
            ('Individual and vehicle characteristics', [
                ('The rider is a female',
                 {'Injury': 'beta_gender_female_I'}),
                ('The rider is a passenger',
                 {'Injury': 'beta_user_category_passenger_I'}),
                ('Age of the individual (years)',
                 {'Injury': 'beta_age_I', 'Fatality': 'beta_age_F'}),
          
            ]),
            ('Collision characteristics', [
                ('Number of vehicles in the accident',
                 {'Injury': 'beta_number_of_involved_vehicles_I'}),
                # Le modele n'a plus qu'un seul terme d'impact arriere, sans
                # distinction de vehicule (cellule 13).
                ('Rear impact on the MMV',
                 {'Injury': 'beta_point_of_impact_back_I'}),
                       ('The rider was not changing direction',
                 {'Injury': 'beta_maneuver_without_change_of_direction_I'}),
            ]),
            ('Infrastructure characteristics', [
                ('Crash occurred on a cycle facility',
                 {'Fatality': 'beta_accident_location_on_cycle_facility_F'}),
                # L'indicatrice est prise BRUTE (= 1 hors intersection) dans les
                # deux utilites : le libelle suit la variable estimee.
                ('Crash occurred away from an intersection',
                 {'Injury': 'beta_intersection_no_intersection_I',
                  'Fatality': 'beta_intersection_no_intersection_F'}),
                ('Posted speed limit at the crash location',
                 {'Injury': 'beta_vma_I',
                  'Fatality': 'beta_vma_F'}),
            ]),
            ('Environment characteristics', [
                # Les deux modalites de nuit sont regroupees en cellule 13 :
                # un seul coefficient, porte par le parametre `..._on_F`.
                ('Crash occurred at night (with or without street lighting)',
                 {'Fatality':
                  'beta_lighting_conditions_night_with_street_lightings_on_F'}),
            ]),
            ('Second-party characteristics', [
                ('A second-party vehicle is a light vehicle',
                 {'Injury': 'beta_vehicle_type_2_light_motorized_vehicle_I',
                  'Fatality': 'beta_vehicle_type_2_light_motorized_vehicle_F'}),
                ('A second-party vehicle is a heavy vehicle',
                 {'Fatality': 'beta_vehicle_type_2_large_motorized_vehicle_F'}),
                ('A second-party vehicle is turning right',
                 {'Fatality': 'beta_maneuver_2_turning_right_F'}),
                ('A second-party vehicle is overtaking',
                 {'Injury': 'beta_maneuver_2_overtaking_I'}),
            ]),
        ],
    },
    'MMV': {
        'results': lambda: (first_defined('results_logit_mmv'),
                            first_defined('results_constant_mmv')),
        'file': 'model_mmv.tex',
        'caption': ('Estimation results of the binary logistic regression '
                    'predicting the injury severity of MMV riders in crashes '
                    'with other MMVs'),
        'label': 'tab:regression_results_mmv',
        'stats_caption': ('Model statistics of the binary logistic regression '
                          'predicting the injury severity of MMV riders in '
                          'crashes with other MMVs'),
        'stats_label': 'tab:regression_stats_mmv',
        'columns': ['Injury'],
        'sections': [
            ('Alternative specific constant', [
                (r'Alternative specific constant $\alpha$',
                 {'Injury': 'constant_I'}),
            ]),
            ('Individual and vehicle characteristics', [
                ('Individual is a female', {'Injury': 'beta_gender_female_I'}),
                ('Age of the rider (years)', {'Injury': 'beta_age_I'}),
            ]),
            ('Collision characteristics', [
                ('The individual was swerving',
                 {'Injury': 'beta_maneuver_swerving_I'}),
                ('The individual was turning left',
                 {'Injury': 'beta_maneuver_turning_left_I'}),
                ('Rear impact with the e-PMD',
                 {'Injury': 'beta_point_of_impact_back_I_epmd'}),
                ('Rear impact with the (e-)bike',
                 {'Injury': 'beta_point_of_impact_back_I_bike'}),
            ]),
            ('Environment characteristics', [
                ('Wet surface', {'Injury': 'beta_surface_condition_wet_I'}),
            ]),
            ('Second-party characteristics', [
                ('At least one second-party driver is a female',
                 {'Injury': 'beta_gender_2_female_I'}),
                ('Mean age of the second-party individuals (years)',
                 {'Injury': 'beta_age_2_I'}),
                ('Second party not identified',
                 {'Injury': 'beta_age_opposite_missing_I'}),
                ('At least one second-party is an e-PMD',
                 {'Injury': 'beta_vehicle_2_e_pmd_I'}),
            ]),
        ],
    },
    'Pedestrian': {
        'results': lambda: (first_defined('results_pedes_mnl'),
                            first_defined('results_pedes_cst')),
        'file': 'model_ped.tex',
        'caption': ('Estimation results of the ordered probit for the injury '
                    'severity of MMV riders and pedestrians in crashes between '
                    'MMVs and pedestrians'),
        'label': 'tab:regression_results_pedes',
        'stats_caption': ('Model statistics of the ordered model predicting the '
                          'injury severity of MMV riders and pedestrians'),
        'stats_label': 'tab:regression_results_pedes_stats',
        'columns': ['Estimates'],
        'sections': [
            ('Individual and vehicle characteristics', [
                ('Individual is a female', {'Estimates': 'beta_gender_female'}),
                ('Individual is a pedestrian',
                 {'Estimates': 'beta_user_category_pedestrian'}),
                ('Age of the individual (years)', {'Estimates': 'beta_age'}),
            ]),
            ('Infrastructure characteristics', [
                # Comme le modele voiture, l'indicatrice est prise BRUTE
                # (= 1 hors intersection) : le libelle suit la variable.
                ('Crash occurred away from an intersection',
                 {'Estimates': 'beta_intersection_no_intersection'}),
                ('Crash occurred at a signalized intersection',
                 {'Estimates': 'beta_crossroad_traffic_lights'}),
            ]),
            ('Second-party characteristics', [
                ('At least one second-party individual is a female',
                 {'Estimates': 'beta_gender_2_female'}),
                ('Mean age of the second-party individuals (years)',
                 {'Estimates': 'beta_age_opposite_mean'}),
                ('Second party not identified',
                 {'Estimates': 'beta_age_opposite_missing'}),
                # Terme d'interaction : le tiers tourne (a gauche ou a droite)
                # ET l'usager observe est le pieton.
                ('The pedestrian was hit by a turning second-party vehicle',
                 {'Estimates': 'beta_maneuver_2_turning'}),
            ]),
        ],
    },
    'Single-vehicle': {
        'results': lambda: (first_defined('results_solo_mnl'),
                            first_defined('results_cst_solo')),
        'file': 'model_solo.tex',
        'caption': ('Estimation results of the ordered probit for the injury '
                    'severity of MMV riders in single-vehicle crashes'),
        'label': 'tab:solo_crashes',
        'stats_caption': ('Model statistics of the ordered probit for the injury '
                          'severity of MMV riders in single-vehicle crashes'),
        'stats_label': 'tab:solo_crashes_stats',
        'columns': ['Estimates'],
        'sections': [
            ('Individual and vehicle characteristics', [
                ('Age of the rider (years)', {'Estimates': 'beta_age'}),
                ('The individual is a passenger',
                 {'Estimates': 'beta_user_category_passenger'}),
                ('Number of passengers',
                 {'Estimates': 'beta_number_of_passengers'}),
            ]),
            ('Infrastructure characteristics', [
                ('Crash occurred on a slope',
                 {'Estimates': 'beta_long_profile_slope'}),
            ]),
        ],
    },
    # --- Les contreparties MNL ------------------------------------------------
    # Memes segments, memes donnees, mais deux utilites au lieu d'un score
    # ordonne : chaque variable recoit un coefficient propre a la blessure et un
    # autre au deces, d'ou deux colonnes. Une variable absente d'une des deux
    # utilites imprime un tiret -- c'est le cas pour la moitie du modele sans
    # tiers, ou 25 deces ne permettent pas d'identifier plus de deux
    # coefficients sur l'alternative mortelle.
    'Pedestrian (MNL)': {
        # LL(c) du logit binaire sur `harmed`, pas celui du probit ordonne a
        # trois niveaux : ce n'est pas la meme variable expliquee.
        'results': lambda: (first_defined('results_pedestrian_mnl',
                                          'results_pedes_mnl'),
                            first_defined('results_pedes_cst_mnl',
                                          'results_pedes_cst')),
        'file': 'model_ped_mnl.tex',
        'caption': ('Estimation results of the logit for the injury severity of '
                    'MMV riders and pedestrians in crashes between MMVs and '
                    'pedestrians'),
        'label': 'tab:regression_results_pedes_mnl',
        'stats_caption': ('Model statistics of the logit predicting the injury '
                          'severity of MMV riders and pedestrians'),
        'stats_label': 'tab:regression_results_pedes_mnl_stats',
        # UNE seule colonne : l'utilite de la blessure et celle du deces sont la
        # meme expression dans la cellule d'estimation, donc il n'existe qu'un
        # jeu de coefficients, suffixes `_I`.
        'columns': ['Estimates'],
        'sections': [
            ('Individual and vehicle characteristics', [
                ('Individual is a female',
                 {'Estimates': 'beta_gender_female_I'}),
                ('Individual is a pedestrian',
                 {'Estimates': 'beta_user_category_pedestrian_I'}),
                ('Age of the individual (years)', {'Estimates': 'beta_age_I'}),
            ]),
            ('Infrastructure characteristics', [
                # ATTENTION au sens : l'utilite porte
                # `(intersection_no_intersection == 0)`, donc l'indicatrice vaut
                # 1 A l'intersection -- l'INVERSE du modele ordonne, qui prend
                # la variable brute.
                ('Crash occurred at an intersection',
                 {'Estimates': 'beta_intersection_no_intersection_I'}),
                ('Crash occurred at a signalized intersection',
                 {'Estimates': 'beta_crossroad_traffic_lights_I'}),
            ]),
            ('Second-party characteristics', [
                ('At least one second-party individual is a female',
                 {'Estimates': 'beta_gender_2_female_I'}),
                ('Mean age of the second-party individuals (years)',
                 {'Estimates': 'beta_age_opposite_mean_I'}),
                ('Second party not identified',
                 {'Estimates': 'beta_age_opposite_missing_I'}),
                       ('The pedestrian was hit by a turning second-party vehicle',
                 {'Estimates': 'beta_maneuver_2_turning'}),
            ]),
            ('Alternative-specific constant', [
                ('Constant', {'Estimates': 'asc_injury_ped'}),
            ]),
        ],
    },
    'Single-vehicle (MNL)': {
        'results': lambda: (first_defined('results_solo_mnl'),
                            first_defined('results_cst_solo_mnl',
                                          'results_cst_solo')),
        'file': 'model_solo_mnl.tex',
        'caption': ('Estimation results of the multinomial logit for the injury '
                    'severity of MMV riders in single-vehicle crashes'),
        'label': 'tab:solo_crashes_mnl',
        'stats_caption': ('Model statistics of the multinomial logit for the '
                          'injury severity of MMV riders in single-vehicle '
                          'crashes'),
        'stats_label': 'tab:solo_crashes_mnl_stats',
        'columns': ['Injury', 'Fatality'],
        'sections': [
            ('Individual and vehicle characteristics', [
                ('Age of the rider (years)',
                 {'Injury': 'beta_age_I', 'Fatality': 'beta_age_F'}),
                ('The individual is a passenger',
                 {'Injury': 'beta_user_category_passenger_I',
                  'Fatality': None}),
                ('The rider was on a standing e-scooter',
                 {'Injury': 'beta_vehicle_e_pmd', 'Fatality': None}),
            ]),
            ('Infrastructure characteristics', [
                ('Crash occurred on a slope',
                 {'Injury': None, 'Fatality': 'beta_long_profile_slope_F'}),
            ]),
            ('Alternative-specific constants', [
                ('Constant',
                 {'Injury': 'asc_injury_sv', 'Fatality': 'asc_fatality_sv'}),
            ]),
        ],
        'notes': [r'A dash marks a variable left out of that utility: with 25 '
                  r'fatalities, the fatal alternative does not support more '
                  r'than two coefficients.'],
    },
}


manuscript_tex = {}
for name, specification in MANUSCRIPT_SPECIFICATIONS.items():
    try:
        results, baseline = specification['results']()
        tex = make_manuscript_table(results, specification, baseline=baseline)
    except Exception as error:
        print(f'{name:16s} : tableau non produit -- '
              f'{type(error).__name__}: {error}')
        continue
    path = MANUSCRIPT_DIRECTORY / specification['file']
    path.write_text(colorize(tex) + '\n', encoding='utf-8')
    manuscript_tex[name] = tex
    print(f'{name:16s} -> {path}')

Car crashes      -> results/manuscript/model_car.tex
MMV              -> results/manuscript/model_mmv.tex
Pedestrian       : tableau non produit -- NameError: aucun de ces resultats n'est defini : results_pedes_mnl
Single-vehicle   -> results/manuscript/model_solo.tex
Pedestrian (MNL) -> results/manuscript/model_ped_mnl.tex
Single-vehicle (MNL) -> results/manuscript/model_solo_mnl.tex


In [39]:
def get_results(file_path):
    """Load a Biogeme `bioResults` object back from its pickle file."""
    with open(file_path, 'rb') as file:
        data = pickle.load(file)
    return res.bioResults(data)




## Out-of sample validation of the models

In [40]:
# Create DataFrames for each year and without each year
years = [2019, 2020, 2021, 2022, 2023]

# Classe pour contenir les données de validation
class ValidationData:
    def __init__(self, estimation, validation):
        self.estimation = estimation
        self.validation = validation

def create_validation_data(df):
    validation_data=[]
    validation_data.append(ValidationData(df[df['Year'].isin([2019, 2020, 2021, 2022])], df[df['Year'] == 2023]))
    df_lyon = df[df['Agglomeration_MÉTROPOLE DE LYON'] == 1]
    df_paris = df[df['Agglomeration_MÉTROPOLE DU GRAND PARIS']==1]
    validation_data.append(ValidationData(df_paris, df_lyon))
    return validation_data



# Create validation data for each dataset
validationData_car = create_validation_data(df_carcrashes)
validationData_mmv= create_validation_data(df_mmv)
validationData_sv_2 = create_validation_data(df_sv)
validationData_pedestrian = create_validation_data(df_pedestrian)

In [41]:
# Validate the model with the validation data for car
validation_results_car = model_car.validate(results_ml_carcrashes, validationData_car)
validation_results_car_cst = model_cst_car.validate(results_ml_carcrashes, validationData_car)


# Initialize variables to store log-likelihoods

loglike_model_car = 0
loglike_constant_car = 0

# Calculate log-likelihood for the model (car)
for i, slide in enumerate(validation_results_car):
    validation_loglike = slide['Loglikelihood'].sum()
    loglike_model_car += validation_loglike
    print(
        f'Log likelihood for {slide.shape[0]} validation data on car (slide {i+1}): '
        f'{validation_loglike}'
    )

# Calculate log-likelihood for the constant model (cars)
for i, slide in enumerate(validation_results_car_cst):
    validation_loglike_cst = slide['Loglikelihood'].sum()
    loglike_constant_car += validation_loglike_cst
    print(
        f'Log likelihood for {slide.shape[0]} validation data on car (constant model, slide {i+1}): '
        f'{validation_loglike_cst}'
    )

# Calculate rho-square for each slide (cars)
for i in range(len(validation_results_car)):
    validation_loglike = validation_results_car[i]['Loglikelihood'].sum()
    validation_loglike_cst = validation_results_car_cst[i]['Loglikelihood'].sum()
    rho_square = 1 - (validation_loglike / validation_loglike_cst)
    print(f'Rho-square for the validation data on car (slide {i+1}): {rho_square}')





Log likelihood for 2129 validation data on car (slide 1): -345.25959388569345
Log likelihood for 1361 validation data on car (slide 2): -227.53072252007894
Log likelihood for 2129 validation data on car (constant model, slide 1): -400.53378858590474
Log likelihood for 1361 validation data on car (constant model, slide 2): -249.84573283135495
Rho-square for the validation data on car (slide 1): 0.1380013279163247
Rho-square for the validation data on car (slide 2): 0.08931515482931451


In [42]:
# Validate the model with the validation data for mmv
validation_results_mmv = model_mmv.validate(results_logit_mmv, validationData_mmv)
validation_results_mmv_cst = model_cst_mmv.validate(results_constant_mmv, validationData_mmv)


# Initialize variables to store log-likelihoods
loglike_model_mmv = 0
loglike_constant_mmv = 0
loglike_model_car = 0
loglike_constant_car = 0

# Calculate log-likelihood for the model (mmv)
for i, slide in enumerate(validation_results_mmv):
    validation_loglike = slide['Loglikelihood'].sum()
    loglike_model_mmv += validation_loglike
    print(
        f'Log likelihood for {slide.shape[0]} validation data on mmv (slide {i+1}): '
        f'{validation_loglike}'
    )

# Calculate log-likelihood for the constant model (mmv)
for i, slide in enumerate(validation_results_mmv_cst):
    validation_loglike_cst = slide['Loglikelihood'].sum()
    loglike_constant_mmv += validation_loglike_cst
    print(
        f'Log likelihood for {slide.shape[0]} validation data on mmv (constant model, slide {i+1}): '
        f'{validation_loglike_cst}'
    )

# Calculate rho-square for each slide (mmv)
for i in range(len(validation_results_mmv)):
    validation_loglike = validation_results_mmv[i]['Loglikelihood'].sum()
    validation_loglike_cst = validation_results_mmv_cst[i]['Loglikelihood'].sum()
    rho_square = 1 - (validation_loglike / validation_loglike_cst)
    print(f'Rho-square for the validation data on mmv (slide {i+1}): {rho_square}')





Log likelihood for 374 validation data on mmv (slide 1): -197.66539392764153
Log likelihood for 88 validation data on mmv (slide 2): -42.342716087938925
Log likelihood for 374 validation data on mmv (constant model, slide 1): -237.09388508888168
Log likelihood for 88 validation data on mmv (constant model, slide 2): -53.872354317967954
Rho-square for the validation data on mmv (slide 1): 0.16629906396134686
Rho-square for the validation data on mmv (slide 2): 0.21401771606227293


In [43]:

# Validate the model with the validation data for mmv
validation_results_pedes = model_pedes_probit.validate(results_pedes_probit, validationData_pedestrian)
validation_results_pedes_cst = model_cst_pedes.validate(results_pedes_cst, validationData_pedestrian)


# Initialize variables to store log-likelihoods
loglike_model_pedes= 0
loglike_constant_pedes = 0

# Calculate log-likelihood for the model (mmv)
for i, slide in enumerate(validation_results_pedes):
    validation_loglike = slide['Loglikelihood'].sum()
    loglike_model_pedes += validation_loglike
    print(
        f'Log likelihood for {slide.shape[0]} validation data on mmv (slide {i+1}): '
        f'{validation_loglike}'
    )

# Calculate log-likelihood for the constant model (mmv)
for i, slide in enumerate(validation_results_pedes_cst):
    validation_loglike_cst = slide['Loglikelihood'].sum()
    loglike_constant_pedes += validation_loglike_cst
    print(
        f'Log likelihood for {slide.shape[0]} validation data on pedes (constant model, slide {i+1}): '
        f'{validation_loglike_cst}'
    )

# Calculate rho-square for each slide (mmv)
for i in range(len(validation_results_pedes)):
    validation_loglike = validation_results_pedes[i]['Loglikelihood'].sum()
    validation_loglike_cst = validation_results_pedes_cst[i]['Loglikelihood'].sum()
    rho_square = 1 - (validation_loglike / validation_loglike_cst)
    print(f'Rho-square for the validation data on pedes (slide {i+1}): {rho_square}')




Log likelihood for 711 validation data on mmv (slide 1): -303.5143579935265
Log likelihood for 223 validation data on mmv (slide 2): -91.050015177684
Log likelihood for 711 validation data on pedes (constant model, slide 1): -491.2867169242943
Log likelihood for 223 validation data on pedes (constant model, slide 2): -151.9994039479823
Rho-square for the validation data on pedes (slide 1): 0.38220524280875867
Rho-square for the validation data on pedes (slide 2): 0.40098439327536173


In [44]:


# Validate the model with the validation data for mmv
validation_results_solo_2 = model_solo_2.validate(results_solo_2, validationData_sv_2)
validation_results_solo_cst = model_cst_solo.validate(results_cst_solo, validationData_sv_2)

# Initialize variables to store log-likelihoods
loglike_model_mmv = 0
loglike_constant_mmv = 0
loglike_model_car = 0
loglike_constant_car = 0

# Calculate log-likelihood for the model (mmv)
for i, slide in enumerate(validation_results_solo_2):
    validation_loglike = slide['Loglikelihood'].sum()
    loglike_model_mmv += validation_loglike
    print(
        f'Log likelihood for {slide.shape[0]} validation data on mmv (slide {i+1}): '
        f'{validation_loglike}'
    )

# Calculate log-likelihood for the constant model (mmv)
for i, slide in enumerate(validation_results_solo_cst):
    validation_loglike_cst = slide['Loglikelihood'].sum()
    loglike_constant_mmv += validation_loglike_cst
    print(
        f'Log likelihood for {slide.shape[0]} validation data on mmv (constant model, slide {i+1}): '
        f'{validation_loglike_cst}'
    )

# Calculate rho-square for each slide (mmv)
for i in range(len(validation_results_solo_2)):
    validation_loglike = validation_results_solo_2[i]['Loglikelihood'].sum()
    validation_loglike_cst = validation_results_solo_cst[i]['Loglikelihood'].sum()
    rho_square = 1 - (validation_loglike / validation_loglike_cst)

    # Calculate rho-square for each slide (mmv
for i in range(len(validation_results_solo_2)):
    validation_loglike = validation_results_solo_2[i]['Loglikelihood'].sum()
    validation_loglike_cst = validation_results_solo_cst[i]['Loglikelihood'].sum()
    rho_square = 1 - (validation_loglike / validation_loglike_cst)
    print(f'Rho-square for the validation data on mmv (slide {i+1}): {rho_square}')


Log likelihood for 422 validation data on mmv (slide 1): -72.9726691923702
Log likelihood for 222 validation data on mmv (slide 2): -70.59772910790414
Log likelihood for 422 validation data on mmv (constant model, slide 1): -83.7974453870486
Log likelihood for 222 validation data on mmv (constant model, slide 2): -76.58265527365617
Rho-square for the validation data on mmv (slide 1): 0.1291778782119225
Rho-square for the validation data on mmv (slide 2): 0.07814989104733738


### Validation tables

The same figures as the loops above, assembled into one table per segment and
written to `results/tables/`: the size of each validation set, the
log-likelihood of the model and of its constants-only counterpart, and the
out-of-sample $1 - LL(\hat{\beta})/LL(c)$.


In [45]:
# --- Out-of-sample validation tables ------------------------------------------
# One table per segment, in the format used in the manuscript: one column per
# validation split, and for each of them the size of the validation set, the
# log-likelihood of the model, that of the constants-only model, and the
# out-of-sample rho-square 1 - LL(beta) / LL(c).
#
# `validate()` re-estimates the model on the estimation part of each split and
# applies it to the validation part, so this cell re-estimates 2 x 2 models per
# segment: it is slow. Called directly on the Biogeme object, it writes its
# intermediate files in the working directory rather than in `results/`.

VALIDATION_TABLES_DIRECTORY = RESULTS_ROOT / 'tables'
VALIDATION_TABLES_DIRECTORY.mkdir(parents=True, exist_ok=True)

# `create_validation_data` builds the splits in this order: (1) estimate on
# 2019-2022, validate on 2023, then (2) estimate on Paris, validate on Lyon.
# The table shows them in the order of the manuscript.
VALIDATION_SLIDES = [(1, 'Lyon vs. Paris'),
                     (0, 'Year 2023 vs. Years 2019--2022')]

SCRATCH_PATTERNS = ('__*.iter', '*.html', '*.pickle')


@contextmanager
def _quiet_validation(*biogeme_objects):
    """Estimate without leaving HTML, pickle or iteration files behind.

    `validate()` re-estimates the model once per split, and each of those fits
    writes `<modelName>.html`, `<modelName>.pickle` and `__<modelName>.iter`
    into the working directory -- which is how the repository root filled up
    with `__*_val_est_*.iter`. The three flags below switch that off; the
    sweep afterwards removes whatever a Biogeme version writes regardless,
    and it only touches files that did not exist before the call.
    """
    flags = ('generate_html', 'generate_pickle', 'save_iterations')
    saved = [(obj, {flag: getattr(obj, flag, None) for flag in flags})
             for obj in biogeme_objects]
    before = {path for pattern in SCRATCH_PATTERNS
              for path in Path.cwd().glob(pattern)}
    try:
        for obj in biogeme_objects:
            for flag in flags:
                if hasattr(obj, flag):
                    setattr(obj, flag, False)
        yield
    finally:
        for obj, previous in saved:
            for flag, value in previous.items():
                if value is not None:
                    setattr(obj, flag, value)
        created = {path for pattern in SCRATCH_PATTERNS
                   for path in Path.cwd().glob(pattern)} - before
        for path in created:
            path.unlink(missing_ok=True)
        if created:
            print(f'  {len(created)} fichier(s) intermediaire(s) supprime(s)')


VALIDATION_ROWS = ['Number of obs. in the validation set',
                   r'\textbf{$LL(\hat{\beta})$}',
                   r'\textbf{$LL(c)$}',
                   r'\textbf{$1 - \frac{LL(\hat{\beta})}{LL(c)}$}']


def validation_table(the_biogeme, results, the_biogeme_constant,
                     results_constant, validation_data, slides=None):
    """Une colonne par decoupage, les quatre lignes du tableau de validation."""
    slides = VALIDATION_SLIDES if slides is None else slides

    with _quiet_validation(the_biogeme, the_biogeme_constant):
        model_slides = the_biogeme.validate(results, validation_data)
        constant_slides = the_biogeme_constant.validate(results_constant,
                                                        validation_data)

    columns = {}
    for position, label in slides:
        model = model_slides[position]
        constant = constant_slides[position]
        log_likelihood = float(model['Loglikelihood'].sum())
        log_likelihood_constant = float(constant['Loglikelihood'].sum())
        columns[label] = [
            model.shape[0],
            log_likelihood,
            log_likelihood_constant,
            1 - log_likelihood / log_likelihood_constant,
        ]
    return pd.DataFrame(columns, index=VALIDATION_ROWS)


def make_validation_table(table, caption=None, label=None,
                          first_width='5.5cm', width='5cm'):
    """Tableau LaTeX de validation, au format du manuscrit."""
    centred = r'>{\centering\arraybackslash}p'
    alignment = (f'{centred}{{{first_width}}}'
                 + f'{centred}{{{width}}}' * len(table.columns))

    lines = [
        r'\begin{table}[H]', r'\small ', r'\centering     ',
        rf'\caption{{{caption}}}' if caption else r'\caption{}',
        rf'\label{{{label}}}' if label else '',
        '',
        rf'\begin{{tabular}}{{{alignment}}}',
        r'\hline',
        '',
        ' & ' + ' & '.join(str(column) for column in table.columns) + r'  \\',
        r'\hline',
    ]

    formats = ['{:.0f}', '{:.0f}', '{:.0f}', '{:.3f}']
    for (name, record), template in zip(table.iterrows(), formats):
        cells = [template.format(float(value)) for value in record]
        lines.append(f'{name} & ' + ' & '.join(cells) + r'\\')

    lines += [r'\hline', r'\end{tabular}', r'\end{table}']
    return '\n'.join(lines)


# (libelle, modele, resultats, modele a constantes, ses resultats,
#  donnees de validation, nom de fichier, legende)
VALIDATION_MODELS = [
    ('Car crashes', model_car, results_ml_carcrashes,
     model_cst_car, results_constant_car, validationData_car, 'val_car',
     "rider's injury severity in crashes with a motor vehicle"),
    ('MMV', model_mmv, results_logit_mmv,
     model_cst_mmv, results_constant_mmv, validationData_mmv, 'val_mmv',
     "rider's injury severity in crashes with a micromobility vehicle"),
    # Pedestrian and single-vehicle are validated on their MNL, each against a
    # constants-only model of its own framework and its own dependent variable.
    # This matters for the pedestrian: its MNL is a binary logit on `harmed`,
    # so referencing it against the three-level ordered-probit baseline would
    # inflate the rho-square.
    ('Pedestrian', model_pedestrian_mnl, results_pedestrian_mnl,
     model_cst_pedes_mnl, results_pedes_cst_mnl, validationData_pedestrian,
     'val_pedes', "injury severity in crashes involving a pedestrian"),
    ('Single-vehicle', model_solo_mnl, results_solo_mnl,
     model_cst_solo_mnl, results_cst_solo_mnl, validationData_sv_2, 'val_solo',
     "rider's injury severity in single-vehicle crashes"),
]

validation_tables, validation_tex = {}, {}
for (name, the_biogeme, results, the_biogeme_constant,
     results_constant, validation_data, stem, subject) in VALIDATION_MODELS:
    print(f'=== {name} ===')
    try:
        table = validation_table(the_biogeme, results,
                                 the_biogeme_constant, results_constant,
                                 validation_data)
    except Exception as error:
        print(f'  validation impossible : {type(error).__name__}: {error}\n')
        continue

    tex = make_validation_table(
        table,
        caption=('Results of the out-of-sample validation for the model '
                 f'predicting {subject}'),
        label=f'tab:{stem}',
    )
    (VALIDATION_TABLES_DIRECTORY / f'{stem}.tex').write_text(
        colorize(tex), encoding='utf-8')
    validation_tables[name], validation_tex[name] = table, tex

    print(table.round(3).to_string())
    print(f'  -> {VALIDATION_TABLES_DIRECTORY / f"{stem}.tex"}\n')

print(next(iter(validation_tex.values()), ''))


# --- The four validations in a single table -----------------------------------
# One block of columns per split, four sub-columns per block -- one per model --
# and the four statistics as rows. More compact than four separate tables, and
# it lets the reader compare the models on the same split at a glance.
MODEL_NUMBERS = {name: f'({position})'
                 for position, name in enumerate(validation_tables, start=1)}


def make_combined_validation_table(
    tables,
    caption=('Out-of-sample validation of the four models. '
             '(1) crashes with a motor vehicle, (2) crashes with another '
             'micromobility vehicle, (3) crashes involving a pedestrian, '
             '(4) single-vehicle crashes.'),
    label='tab:validation_all',
):
    models = list(tables)
    splits = [label_text for _, label_text in VALIDATION_SLIDES]
    rows = VALIDATION_ROWS
    width = 1 + len(splits) * len(models)

    # `{|c}` reproduit le filet vertical du preambule dans l'en-tete groupe.
    group_header = _row([''] + [r'\multicolumn{' + str(len(models))
                                + r'}{|c}{\textbf{' + split + '}}'
                                for split in splits])
    rules = ''.join(rf'\cline{{{2 + position * len(models)}-'
                    rf'{1 + (position + 1) * len(models)}}}'
                    for position in range(len(splits)))
    model_header = _row([r'\textbf{Statistic}']
                        + [MODEL_NUMBERS[name] for _ in splits for name in models])
    alignment = ('p{5.5cm}'
                 + ('|' + 'r' * len(models)) * len(splits))

    lines = [
        r'\begin{table}[H]', r'\small', r'\centering',
        r'\setlength{\tabcolsep}{4pt}',
        rf'\caption{{{caption}}}', rf'\label{{{label}}}',
        rf'\begin{{tabular}}{{{alignment}}}',
        r'\hline', group_header, rules, model_header, r'\hline',
    ]

    formats = ['{:.0f}', '{:.0f}', '{:.0f}', '{:.3f}']
    for row, template in zip(rows, formats):
        cells = []
        for split in splits:
            for name in models:
                table = tables[name]
                value = table[split][row] if split in table.columns else None
                cells.append('' if value is None else template.format(float(value)))
        lines.append(_row([row] + cells))

    lines += [r'\hline', r'\end{tabular}', r'\end{table}']
    return '\n'.join(lines)


combined_validation_tex = make_combined_validation_table(validation_tables)
combined_validation_path = VALIDATION_TABLES_DIRECTORY / 'val_all.tex'
combined_validation_path.write_text(colorize(combined_validation_tex) + '\n',
                                    encoding='utf-8')
print()
print(combined_validation_path)
print(combined_validation_tex)

=== Car crashes ===
  4 fichier(s) intermediaire(s) supprime(s)
                                              Lyon vs. Paris  Year 2023 vs. Years 2019--2022
Number of obs. in the validation set                1361.000                        2129.000
\textbf{$LL(\hat{\beta})$}                          -227.531                        -345.260
\textbf{$LL(c)$}                                    -249.846                        -400.534
\textbf{$1 - \frac{LL(\hat{\beta})}{LL(c)}$}           0.089                           0.138
  -> results/tables/val_car.tex

=== MMV ===
  4 fichier(s) intermediaire(s) supprime(s)
                                              Lyon vs. Paris  Year 2023 vs. Years 2019--2022
Number of obs. in the validation set                  88.000                         374.000
\textbf{$LL(\hat{\beta})$}                           -42.343                        -197.665
\textbf{$LL(c)$}                                     -53.872                        -237.094
\textbf{$1